In [1]:

############################################################
# Spatiotemporal fitness AI analysis in Python
# PCA, Tensor decomposition, VAE
#
# Input format expected:
# Gene, Time, Space, logFC
#
# Main biological question:
# When, where, and under what host-context transition
# does each gene become important?
############################################################

import os
import warnings

# Resource controls.
# n_cores controls explicit Python/Torch parallel work; BLAS threads stay at 1 to avoid nested oversubscription.
n_cores = min(4, os.cpu_count() or 1)
blas_threads = 1
os.environ["OMP_NUM_THREADS"] = str(blas_threads)
os.environ["MKL_NUM_THREADS"] = str(blas_threads)
os.environ["OPENBLAS_NUM_THREADS"] = str(blas_threads)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(blas_threads)
os.environ["NUMEXPR_NUM_THREADS"] = str(blas_threads)
import numpy as np
import pandas as pd
import textwrap

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc

try:
    from joblib import Parallel, delayed
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False
    warnings.warn("joblib is not installed. Parallel tensor rank scans will run serially.")

############################################################
# Optional packages
############################################################

try:
    import tensorly as tl
    from tensorly.decomposition import parafac
    from tensorly.cp_tensor import cp_to_tensor
    HAS_TENSORLY = True
except ImportError:
    HAS_TENSORLY = False
    warnings.warn("tensorly is not installed. Tensor decomposition will be skipped.")

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader, random_split
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    warnings.warn("torch is not installed. VAE will be skipped.")

# %% Cell 3
############################################################
# 1. User parameters
############################################################

try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()
fallback_project_dir = "/Users/sidaye/Documents/python/ST_MultiCAST"

if os.path.exists(os.path.join(script_dir, "Input")):
    project_dir = script_dir
elif os.path.exists(os.path.join(fallback_project_dir, "Input")):
    project_dir = fallback_project_dir
else:
    project_dir = script_dir

input_file = os.path.join(project_dir, "Input", "Spatial_temporal_MultiSCAST_FC_final_capping.csv")
annotation_file = os.path.join(project_dir, "Input", "new_annotations_with_uniprot_names.csv")
putative_tf_file = os.path.join(project_dir, "Input", "putative_transcription_regulators.xlsx")

outdir = os.path.join(project_dir, "Output", "AI_spatiotemporal_models_python")
os.makedirs(outdir, exist_ok=True)

Spacepoints = [
    "st", "SI1", "SI2", "SI3", "SI4", "SI5",
    "SI6", "SI7", "SI8", "SI9", "ce", "co"
]

Full_Timepoints = ["1h", "3h", "6h", "12h", "24h"]

feature_order = [
    f"{time}_{space}"
    for time in Full_Timepoints
    for space in Spacepoints
]

# TRUE-like:
# compares profile shape across genes.
# FALSE-like:
# preserves absolute fitness magnitude.
scale_by_gene = True

print(f"Using up to {n_cores} CPU cores for explicit parallel work; BLAS threads per worker = {blas_threads}.")

# Tensor decomposition rank.
# For a fair reconstruction comparison with PCA-5, use a fixed CP rank = 5.
# Set auto_select_tensor_rank=True only if you want the script to choose rank by elbow.
tensor_rank = 5
auto_select_tensor_rank = False
tensor_rank_candidates = list(range(1, 13))

# VAE parameters
run_vae = True
vae_latent_dim = 4  # main VAE latent dimension for VAE result display
vae_epochs = 80
vae_batch_size = 256
vae_learning_rate = 1e-3
vae_hidden_1 = 64
vae_hidden_2 = 32
vae_random_seed = 1

if HAS_TORCH:
    torch.set_num_threads(n_cores)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

# %% Cell 4
############################################################
# 2. Helper functions
############################################################

def savefig(path, width=7, height=5, dpi=300):
    plt.gcf().set_size_inches(width, height)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight", transparent=True)
    plt.close()


def parse_feature(feature):
    """
    Parse feature such as '12h_SI6' into Time='12h', Space='SI6'.
    """
    parts = feature.split("_", 1)
    if len(parts) != 2:
        return None, None
    return parts[0], parts[1]


def load_gene_name_map(annotation_path):
    """
    Load locus_ID -> gene_name mapping.
    Empty or missing gene_name values fall back to the original locus_ID.
    """
    if not os.path.exists(annotation_path):
        warnings.warn(
            f"Annotation file not found: {annotation_path}. "
            "Figures will use locus IDs."
        )
        return {}

    annot = pd.read_csv(annotation_path)
    required = {"locus_ID", "gene_name"}
    missing = required - set(annot.columns)

    if missing:
        warnings.warn(
            f"Annotation file is missing columns {sorted(missing)}. "
            "Figures will use locus IDs."
        )
        return {}

    annot = annot[["locus_ID", "gene_name"]].copy()
    annot["locus_ID"] = annot["locus_ID"].astype(str).str.strip()
    annot["gene_name"] = annot["gene_name"].astype("string").str.strip()
    annot = annot[
        annot["locus_ID"].notna()
        & annot["gene_name"].notna()
        & (annot["gene_name"] != "")
        & (annot["gene_name"].str.lower() != "nan")
    ]

    return dict(zip(annot["locus_ID"], annot["gene_name"]))


def display_gene(gene):
    """
    Return gene_name when available; otherwise return the original locus ID.
    """
    gene = str(gene)
    return gene_name_map.get(gene, gene)


def wrap_label(label, width=18):
    """
    Wrap long gene labels so subplot titles do not collide.
    """
    return "\n".join(textwrap.wrap(str(label), width=width)) or str(label)


def add_gene_display(df, gene_col="Gene", display_col="Gene_display"):
    """
    Add a plotting-friendly gene label column without replacing locus IDs.
    """
    df = df.copy()
    df[display_col] = df[gene_col].map(display_gene)
    return df


def annotate_gene_points(ax, df, x_col, y_col, label_col="Gene_display", fontsize=3.5):
    """
    Label scatterplot points with gene_name when available, otherwise locus_ID.
    """
    if label_col not in df.columns:
        labels = df["Gene"].map(display_gene)
    else:
        labels = df[label_col].fillna(df["Gene"].map(display_gene))

    for (_, row), label in zip(df.iterrows(), labels):
        if pd.notna(row[x_col]) and pd.notna(row[y_col]):
            ax.text(
                row[x_col],
                row[y_col],
                str(label),
                fontsize=fontsize,
                alpha=0.65
            )


def choose_tensor_rank_from_mse(rank_mse_df):
    """
    Pick an elbow rank from reconstruction MSE values.
    The selected rank is the point with maximum distance below the line
    connecting the first and last rank-MSE points.
    """
    rank_mse_df = rank_mse_df.sort_values("Rank").reset_index(drop=True)

    if len(rank_mse_df) == 1:
        return int(rank_mse_df["Rank"].iloc[0])

    if len(rank_mse_df) == 2:
        return int(rank_mse_df.loc[rank_mse_df["Reconstruction_MSE"].idxmin(), "Rank"])

    x = rank_mse_df["Rank"].to_numpy(dtype=float)
    y = rank_mse_df["Reconstruction_MSE"].to_numpy(dtype=float)

    x_norm = (x - x.min()) / (x.max() - x.min())
    y_range = y.max() - y.min()
    if y_range == 0:
        return int(rank_mse_df["Rank"].iloc[0])

    y_norm = (y - y.min()) / y_range
    start = np.array([x_norm[0], y_norm[0]])
    end = np.array([x_norm[-1], y_norm[-1]])
    line = end - start
    line_norm = np.linalg.norm(line)

    if line_norm == 0:
        return int(rank_mse_df.loc[rank_mse_df["Reconstruction_MSE"].idxmin(), "Rank"])

    points = np.column_stack([x_norm, y_norm])
    distances = np.abs(np.cross(line, start - points) / line_norm)
    return int(rank_mse_df.loc[np.argmax(distances), "Rank"])


def select_tensor_rank_by_reconstruction_mse(tensor_array, candidate_ranks, outfile_prefix):
    """
    Fit CP decompositions across candidate ranks, save reconstruction MSE,
    and return the elbow-selected rank.
    """
    def fit_rank(rank):
        rank = int(rank)
        rank_result = parafac(
            tensor_array,
            rank=rank,
            n_iter_max=250,
            tol=1e-5,
            init="svd",
            random_state=1,
            normalize_factors=False
        )
        rank_reconstructed = cp_to_tensor(rank_result)
        rank_mse = np.mean((tensor_array - rank_reconstructed) ** 2)
        return {
            "Rank": rank,
            "Reconstruction_MSE": rank_mse
        }

    if HAS_JOBLIB and n_cores > 1 and len(candidate_ranks) > 1:
        records = Parallel(n_jobs=n_cores)(
            delayed(fit_rank)(rank)
            for rank in candidate_ranks
        )
    else:
        records = [fit_rank(rank) for rank in candidate_ranks]

    rank_mse_df = pd.DataFrame(records)
    selected_rank = choose_tensor_rank_from_mse(rank_mse_df)
    rank_mse_df["Selected"] = rank_mse_df["Rank"] == selected_rank

    rank_mse_df.to_csv(outfile_prefix + ".csv", index=False)

    plt.figure()
    plt.plot(
        rank_mse_df["Rank"],
        rank_mse_df["Reconstruction_MSE"],
        marker="o"
    )
    selected_row = rank_mse_df[rank_mse_df["Selected"]].iloc[0]
    plt.scatter(
        [selected_row["Rank"]],
        [selected_row["Reconstruction_MSE"]],
        s=80,
        color="red",
        zorder=3,
        label=f"Selected rank = {selected_rank}"
    )
    plt.xlabel("Tensor rank")
    plt.ylabel("Reconstruction MSE")
    plt.title("Tensor rank selection by reconstruction MSE")
    plt.legend()
    savefig(outfile_prefix + ".pdf", width=7, height=4)

    return selected_rank, rank_mse_df


def make_gene_summary(X_raw_df):
    """
    X_raw_df:
        rows = genes
        columns = 5 × 12 features
    """
    records = []

    for gene, row in X_raw_df.iterrows():
        values = row.values.astype(float)

        min_idx = np.nanargmin(values)
        max_idx = np.nanargmax(values)

        min_feature = X_raw_df.columns[min_idx]
        max_feature = X_raw_df.columns[max_idx]

        min_time, min_space = parse_feature(min_feature)
        max_time, max_space = parse_feature(max_feature)

        records.append({
            "Gene": gene,
            "Gene_display": display_gene(gene),
            "mean_fitness": np.nanmean(values),
            "min_fitness": np.nanmin(values),
            "max_fitness": np.nanmax(values),
            "strongest_defect_feature": min_feature,
            "strongest_defect_time": min_time,
            "strongest_defect_space": min_space,
            "strongest_positive_feature": max_feature,
            "strongest_positive_time": max_time,
            "strongest_positive_space": max_space,
            "dynamic_range": np.nanmax(values) - np.nanmin(values)
        })

    return pd.DataFrame(records)


def matrix_to_long(X_df, value_name="value"):
    """
    Convert gene × feature matrix to long dataframe:
    Gene, Feature, Time, Space, value
    """
    long_df = (
        X_df
        .reset_index()
        .rename(columns={"index": "Gene"})
        .melt(id_vars="Gene", var_name="Feature", value_name=value_name)
    )

    long_df["Gene_display"] = long_df["Gene"].map(display_gene)

    ts = long_df["Feature"].apply(lambda x: pd.Series(parse_feature(x)))
    ts.columns = ["Time", "Space"]

    long_df = pd.concat([long_df, ts], axis=1)
    long_df["Time"] = pd.Categorical(long_df["Time"], categories=Full_Timepoints, ordered=True)
    long_df["Space"] = pd.Categorical(long_df["Space"], categories=Spacepoints, ordered=True)

    return long_df


def plot_gene_matrix_heatmap(long_df, title, subtitle, outfile):
    """
    long_df should have:
    Gene, Time, Space, value, Type
    """
    genes = long_df["Gene"].unique()
    types = long_df["Type"].unique()

    n_rows = len(genes)
    n_cols = len(types)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.6 * n_cols, 2.35 * n_rows + 1.0),
        squeeze=False
    )

    vmin = long_df["value"].quantile(0.02)
    vmax = long_df["value"].quantile(0.98)

    for i, gene in enumerate(genes):
        for j, typ in enumerate(types):
            ax = axes[i, j]
            sub = long_df[(long_df["Gene"] == gene) & (long_df["Type"] == typ)]
            gene_label = (
                sub["Gene_display"].iloc[0]
                if "Gene_display" in sub.columns and not sub.empty
                else display_gene(gene)
            )

            mat = (
                sub.pivot(index="Time", columns="Space", values="value")
                .reindex(index=Full_Timepoints, columns=Spacepoints)
            )

            im = ax.imshow(mat.values, aspect="auto", vmin=vmin, vmax=vmax)

            ax.set_title(f"{wrap_label(gene_label)}\n{typ}", fontsize=7, pad=6)
            ax.set_xticks(range(len(Spacepoints)))
            ax.set_xticklabels(Spacepoints, rotation=45, ha="right", fontsize=7)
            ax.set_yticks(range(len(Full_Timepoints)))
            ax.set_yticklabels(Full_Timepoints, fontsize=7)

    fig.suptitle(title + "\n" + subtitle, fontsize=11, y=0.985)
    fig.subplots_adjust(top=0.88, bottom=0.11, hspace=0.7, wspace=0.25)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.55, label="Scaled fitness")

    plt.savefig(outfile, dpi=300, bbox_inches="tight", pad_inches=0.25, transparent=True)
    plt.close()

# %% Cell 5
############################################################
# 3. Load and preprocess data
############################################################

gene_name_map = load_gene_name_map(annotation_file)
print("Gene names loaded from annotation:", len(gene_name_map))

df_plot = pd.read_csv(input_file)
print("Raw data head:")
print(df_plot.head())
print("Raw data shape:", df_plot.shape)

# %% Cell 6
required_cols = ["Gene", "Time", "Space", "logFC"]
missing_cols = [c for c in required_cols if c not in df_plot.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_clean = df_plot.copy()

# Important:
# Keep Time and Space as strings during groupby.
# Do NOT convert to categorical before groupby.
df_clean["Gene"] = df_clean["Gene"].astype(str)
df_clean["Time"] = df_clean["Time"].astype(str)
df_clean["Space"] = df_clean["Space"].astype(str)

# Keep only expected timepoints and spaces.
df_clean = df_clean[
    df_clean["Time"].isin(Full_Timepoints) &
    df_clean["Space"].isin(Spacepoints)
].copy()
df_clean["Feature"] = df_clean["Time"] + "_" + df_clean["Space"]

# Aggregate duplicated Gene-Time-Space rows if present.
# Because Time/Space are strings here, pandas will not expand unobserved categorical combinations.
df_clean = (
    df_clean
    .groupby(["Gene", "Time", "Space", "Feature"], as_index=False)
    .agg(logFC=("logFC", "mean"))
)

print("Cleaned data shape:", df_clean.shape)
print(df_clean.head())

# %% Cell 7
############################################################
# 4. Build gene × 60 matrix
############################################################

wide_df = (
    df_clean
    .pivot_table(
        index="Gene",
        columns="Feature",
        values="logFC",
        aggfunc="mean"
    )
)

existing_features = [f for f in feature_order if f in wide_df.columns]
missing_features = [f for f in feature_order if f not in wide_df.columns]

if missing_features:
    warnings.warn(
        "Some expected 5×12 features are missing from the input: "
        + ", ".join(missing_features)
    )

wide_df = wide_df[existing_features]

# Keep only genes with complete profiles across existing expected features.
wide_complete = wide_df.dropna(axis=0, how="any")

X_raw_df = wide_complete.copy()
gene_ids = X_raw_df.index.to_list()

print("Number of genes with complete profiles:", X_raw_df.shape[0])
print("Number of features:", X_raw_df.shape[1])

if X_raw_df.shape[1] != 60:
    warnings.warn(
        f"Expected 60 features, but found {X_raw_df.shape[1]}. "
        "The script will continue with the available features."
    )

# %% Cell 8
############################################################
# 5. Scaling
############################################################

X_raw = X_raw_df.values.astype(float)

if scale_by_gene:
    row_mean = X_raw.mean(axis=1, keepdims=True)
    row_std = X_raw.std(axis=1, keepdims=True)
    row_std[row_std == 0] = 1.0
    X_scaled = (X_raw - row_mean) / row_std
else:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

X_scaled = np.nan_to_num(X_scaled)

X_scaled_df = pd.DataFrame(
    X_scaled,
    index=gene_ids,
    columns=X_raw_df.columns
)

gene_summary = make_gene_summary(X_raw_df)
gene_summary.to_csv(
    os.path.join(outdir, "gene_spatiotemporal_summary.csv"),
    index=False
)

# %% Cell 9
############################################################
# 6. PCA analysis
############################################################

pca = PCA()
pca_scores_full = pca.fit_transform(X_scaled)

n_pcs_to_save = min(10, pca_scores_full.shape[1])

pca_var = pd.DataFrame({
    "PC": [f"PC{i + 1}" for i in range(len(pca.explained_variance_ratio_))],
    "variance_explained": pca.explained_variance_ratio_,
    "cumulative_variance": np.cumsum(pca.explained_variance_ratio_)
})

pca_scores = pd.DataFrame(
    pca_scores_full[:, :n_pcs_to_save],
    columns=[f"PC{i + 1}" for i in range(n_pcs_to_save)]
)
pca_scores["Gene"] = gene_ids
pca_scores = pca_scores.merge(gene_summary, on="Gene", how="left")

pca_scores.to_csv(
    os.path.join(outdir, "PCA_gene_scores.csv"),
    index=False
)

pca_var.to_csv(
    os.path.join(outdir, "PCA_variance_explained.csv"),
    index=False
)

# PCA variance plot
plt.figure()
n_show = min(20, len(pca_var))
plt.bar(pca_var["PC"].iloc[:n_show], pca_var["variance_explained"].iloc[:n_show])
plt.xticks(rotation=45, ha="right")
plt.xlabel("Principal component")
plt.ylabel("Fraction of variance explained")
plt.title("PCA variance explained")
savefig(
    os.path.join(outdir, "PCA_variance_explained.pdf"),
    width=7,
    height=4
)

# PCA gene map colored by strongest defect space
plt.figure()
for space in Spacepoints:
    sub = pca_scores[pca_scores["strongest_defect_space"] == space]
    plt.scatter(sub["PC1"], sub["PC2"], s=10, alpha=0.75, label=space)

#annotate_gene_points(plt.gca(), pca_scores, "PC1", "PC2")
plt.xlabel(f"PC1: {100 * pca_var['variance_explained'].iloc[0]:.1f}%")
plt.ylabel(f"PC2: {100 * pca_var['variance_explained'].iloc[1]:.1f}%")
plt.title(
    "PCA of 5 × 12 spatiotemporal fitness profiles\n"
    "Each point is one gene; color = location of strongest fitness defect"
)
plt.legend(title="Strongest defect space", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
savefig(
    os.path.join(outdir, "PCA_gene_map_by_strongest_defect_space.pdf"),
    width=8,
    height=5
)

# PCA gene map colored by strongest defect time
plt.figure()
for time in Full_Timepoints:
    sub = pca_scores[pca_scores["strongest_defect_time"] == time]
    plt.scatter(sub["PC1"], sub["PC2"], s=10, alpha=0.75, label=time)

#annotate_gene_points(plt.gca(), pca_scores, "PC1", "PC2")
plt.xlabel(f"PC1: {100 * pca_var['variance_explained'].iloc[0]:.1f}%")
plt.ylabel(f"PC2: {100 * pca_var['variance_explained'].iloc[1]:.1f}%")
plt.title(
    "PCA of 5 × 12 spatiotemporal fitness profiles\n"
    "Each point is one gene; color = time of strongest fitness defect"
)
plt.legend(title="Strongest defect time", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
savefig(
    os.path.join(outdir, "PCA_gene_map_by_strongest_defect_time.pdf"),
    width=8,
    height=5
)

# %% Cell 10
############################################################
# 7. Tensor decomposition
############################################################
# Question:
# What gene × time × GI-space programs explain the fitness landscape?
############################################################

tensor_recon_mse = np.nan
cp_result = None

# Tensor requires full 5×12 feature set.
can_run_tensor = HAS_TENSORLY and all(f in X_scaled_df.columns for f in feature_order)

if HAS_TENSORLY and not can_run_tensor:
    warnings.warn("Tensor decomposition skipped because not all 60 time-space features are present.")

if can_run_tensor:

    tl.set_backend("numpy")

    G = len(gene_ids)
    Tn = len(Full_Timepoints)
    Sn = len(Spacepoints)

    tensor_array = np.zeros((G, Tn, Sn), dtype=float)

    for ti, time in enumerate(Full_Timepoints):
        for si, space in enumerate(Spacepoints):
            feat = f"{time}_{space}"
            tensor_array[:, ti, si] = X_scaled_df[feat].values

    if auto_select_tensor_rank:
        valid_tensor_rank_candidates = [
            int(rank)
            for rank in tensor_rank_candidates
            if int(rank) >= 1
        ]
        if not valid_tensor_rank_candidates:
            raise ValueError("tensor_rank_candidates must contain at least one positive rank.")

        tensor_rank, tensor_rank_selection_df = select_tensor_rank_by_reconstruction_mse(
            tensor_array,
            valid_tensor_rank_candidates,
            os.path.join(outdir, "Tensor_rank_selection_reconstruction_MSE")
        )
        print(f"Selected tensor rank from reconstruction MSE elbow: {tensor_rank}")

    cp_result = parafac(
        tensor_array,
        rank=tensor_rank,
        n_iter_max=500,
        tol=1e-7,
        init="svd",
        random_state=1,
        normalize_factors=False
    )

    weights, factors = cp_result
    gene_factor, time_factor, space_factor = factors

    # Absorb weights into gene factor for easier interpretation.
    gene_factor = gene_factor * weights.reshape(1, -1)

    component_names = [f"TensorComp{i + 1}" for i in range(tensor_rank)]

    gene_factor_df = pd.DataFrame(gene_factor, columns=component_names)
    gene_factor_df["Gene"] = gene_ids
    gene_factor_df = gene_factor_df.merge(gene_summary, on="Gene", how="left")

    time_factor_df = pd.DataFrame(time_factor, columns=component_names)
    time_factor_df["Time"] = Full_Timepoints

    space_factor_df = pd.DataFrame(space_factor, columns=component_names)
    space_factor_df["Space"] = Spacepoints

    gene_factor_df.to_csv(
        os.path.join(outdir, "Tensor_gene_loadings.csv"),
        index=False
    )

    time_factor_df.to_csv(
        os.path.join(outdir, "Tensor_time_loadings.csv"),
        index=False
    )

    space_factor_df.to_csv(
        os.path.join(outdir, "Tensor_space_loadings.csv"),
        index=False
    )

    tensor_gene_long = (
        gene_factor_df[["Gene"] + component_names]
        .melt(id_vars="Gene", var_name="Component", value_name="Gene_loading")
        .merge(gene_summary, on="Gene", how="left")
    )

    tensor_gene_long["abs_loading"] = tensor_gene_long["Gene_loading"].abs()
    tensor_gene_long["abs_rank"] = (
        tensor_gene_long
        .groupby("Component")["abs_loading"]
        .rank(method="first", ascending=False)
    )

    tensor_gene_long.to_csv(
        os.path.join(outdir, "Tensor_gene_loadings_long.csv"),
        index=False
    )

    top_tensor_genes = (
        tensor_gene_long
        .sort_values(["Component", "abs_loading"], ascending=[True, False])
        .groupby("Component")
        .head(50)
        .reset_index(drop=True)
    )

    top_tensor_genes.to_csv(
        os.path.join(outdir, "Tensor_top50_genes_per_component.csv"),
        index=False
    )

    tensor_time_long = (
        time_factor_df
        .melt(id_vars="Time", var_name="Component", value_name="Time_loading")
    )

    tensor_space_long = (
        space_factor_df
        .melt(id_vars="Space", var_name="Component", value_name="Space_loading")
    )

    ########################################################
    # 7.1 Tensor temporal programs
    ########################################################

    plt.figure()
    for comp in component_names:
        sub = tensor_time_long[tensor_time_long["Component"] == comp].copy()
        sub["Time"] = pd.Categorical(
            sub["Time"],
            categories=Full_Timepoints,
            ordered=True
        )
        sub = sub.sort_values("Time")
        plt.plot(sub["Time"].astype(str), sub["Time_loading"], marker="o", label=comp)

    plt.xlabel("Time")
    plt.ylabel("Temporal loading")
    plt.title(
        "Tensor decomposition: temporal programs\n"
        "Question: when does each hidden fitness program become important?"
    )
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    savefig(
        os.path.join(outdir, "Tensor_temporal_programs.pdf"),
        width=8,
        height=4.5
    )

    ########################################################
    # 7.2 Tensor spatial programs
    ########################################################

    plt.figure()
    for comp in component_names:
        sub = tensor_space_long[tensor_space_long["Component"] == comp].copy()
        sub["Space"] = pd.Categorical(
            sub["Space"],
            categories=Spacepoints,
            ordered=True
        )
        sub = sub.sort_values("Space")
        plt.plot(sub["Space"].astype(str), sub["Space_loading"], marker="o", label=comp)

    plt.xlabel("GI location")
    plt.ylabel("Spatial loading")
    plt.xticks(rotation=45, ha="right")
    plt.title(
        "Tensor decomposition: GI spatial programs\n"
        "Question: where in the GI tract does each hidden fitness program matter?"
    )
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    savefig(
        os.path.join(outdir, "Tensor_spatial_programs.pdf"),
        width=8,
        height=4.5
    )

    ########################################################
    # 7.3 Top genes per tensor component
    ########################################################

    top_plot = (
        top_tensor_genes
        .sort_values(["Component", "abs_loading"], ascending=[True, False])
        .groupby("Component")
        .head(15)
        .reset_index(drop=True)
    )

    n_comp = len(component_names)
    ncols = 3
    nrows = int(np.ceil(n_comp / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.6 * nrows))
    axes = np.array(axes).reshape(-1)

    for i, comp in enumerate(component_names):
        ax = axes[i]
        sub = top_plot[top_plot["Component"] == comp].copy()
        sub = sub.sort_values("Gene_loading")

        ax.barh(sub["Gene_display"], sub["Gene_loading"])
        ax.set_title(comp, pad=10)
        ax.set_xlabel("Tensor gene loading", labelpad=8)
        ax.tick_params(axis="y", labelsize=7)

    for j in range(len(component_names), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        "Top genes driving each tensor component\n"
        "Question: which genes define each spatiotemporal fitness program?",
        fontsize=13,
        y=0.98
    )
    fig.subplots_adjust(hspace=0.85, wspace=0.35, top=0.82, bottom=0.12)

    plt.savefig(
        os.path.join(outdir, "Tensor_top_genes_per_component.pdf"),
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )
    plt.close()

    ########################################################
    # 7.4 Tensor component 5 × 12 heatmaps
    ########################################################

    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.8 * nrows))
    axes = np.array(axes).reshape(-1)

    all_component_mats = []

    for k, comp in enumerate(component_names):
        mat = np.outer(time_factor[:, k], space_factor[:, k])
        all_component_mats.append(mat)

    all_values = np.concatenate([m.flatten() for m in all_component_mats])
    vmin = np.quantile(all_values, 0.02)
    vmax = np.quantile(all_values, 0.98)

    im = None

    for k, comp in enumerate(component_names):
        ax = axes[k]
        mat = all_component_mats[k]

        im = ax.imshow(mat, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set_title(comp, pad=10)
        ax.set_xticks(range(len(Spacepoints)))
        ax.set_xticklabels(Spacepoints, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(Full_Timepoints)))
        ax.set_yticklabels(Full_Timepoints, fontsize=8)
        ax.set_xlabel("GI location", labelpad=8)
        ax.set_ylabel("Time")

    for j in range(len(component_names), len(axes)):
        axes[j].axis("off")

    fig.suptitle(
        "Tensor component 5 × 12 spatiotemporal landscapes\n"
        "Question: what host-context pattern does each component represent?",
        fontsize=13,
        y=0.98
    )
    fig.subplots_adjust(hspace=0.85, wspace=0.35, top=0.82, bottom=0.12)

    fig.colorbar(
        im,
        ax=axes.ravel().tolist(),
        shrink=0.6,
        label="Time × Space loading"
    )

    plt.savefig(
        os.path.join(outdir, "Tensor_component_spatiotemporal_heatmaps.pdf"),
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )
    plt.close()

    ########################################################
    # 7.5 Tensor reconstruction MSE
    ########################################################

    tensor_reconstructed = cp_to_tensor(cp_result)

    tensor_recon_matrix = np.zeros_like(X_scaled)

    for ti, time in enumerate(Full_Timepoints):
        for si, space in enumerate(Spacepoints):
            feat = f"{time}_{space}"
            col_idx = X_scaled_df.columns.get_loc(feat)
            tensor_recon_matrix[:, col_idx] = tensor_reconstructed[:, ti, si]

    tensor_recon_mse = np.mean((X_scaled - tensor_recon_matrix) ** 2)

else:
    print("Tensor decomposition skipped.")

# %% Cell 11
############################################################
# 8. VAE analysis using PyTorch
############################################################

vae_recon_mse = np.nan

if run_vae and HAS_TORCH:

    torch.manual_seed(vae_random_seed)
    np.random.seed(vae_random_seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device for VAE:", device)

    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

    dataset = TensorDataset(X_tensor)

    n_total = len(dataset)
    n_val = max(1, int(0.15 * n_total))
    n_train = n_total - n_val

    train_dataset, val_dataset = random_split(
        dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(vae_random_seed)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=vae_batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=vae_batch_size,
        shuffle=False
    )

    class VAE(nn.Module):
        def __init__(self, input_dim, hidden_1=64, hidden_2=32, latent_dim=2):
            super().__init__()

            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden_1),
                nn.ReLU(),
                nn.Linear(hidden_1, hidden_2),
                nn.ReLU()
            )

            self.z_mean = nn.Linear(hidden_2, latent_dim)
            self.z_log_var = nn.Linear(hidden_2, latent_dim)

            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, hidden_2),
                nn.ReLU(),
                nn.Linear(hidden_2, hidden_1),
                nn.ReLU(),
                nn.Linear(hidden_1, input_dim)
            )

        def encode(self, x):
            h = self.encoder(x)
            return self.z_mean(h), self.z_log_var(h)

        def reparameterize(self, mean, log_var):
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mean + eps * std

        def decode(self, z):
            return self.decoder(z)

        def forward(self, x):
            mean, log_var = self.encode(x)
            z = self.reparameterize(mean, log_var)
            recon = self.decode(z)
            return recon, mean, log_var

    def vae_loss_function(recon_x, x, mean, log_var):
        recon_loss = nn.functional.mse_loss(recon_x, x, reduction="sum")
        kl_loss = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
        return recon_loss + kl_loss

    input_dim = X_scaled.shape[1]

    vae = VAE(
        input_dim=input_dim,
        hidden_1=vae_hidden_1,
        hidden_2=vae_hidden_2,
        latent_dim=vae_latent_dim
    ).to(device)

    optimizer = optim.Adam(vae.parameters(), lr=vae_learning_rate)

    history = []

    for epoch in range(1, vae_epochs + 1):
        vae.train()
        train_loss = 0.0

        for batch in train_loader:
            x_batch = batch[0].to(device)

            optimizer.zero_grad()
            recon_batch, mean, log_var = vae(x_batch)
            loss = vae_loss_function(recon_batch, x_batch, mean, log_var)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= n_train

        vae.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                x_batch = batch[0].to(device)
                recon_batch, mean, log_var = vae(x_batch)
                loss = vae_loss_function(recon_batch, x_batch, mean, log_var)
                val_loss += loss.item()

        val_loss /= n_val

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        })

        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        os.path.join(outdir, "VAE_training_history.csv"),
        index=False
    )

    plt.figure()
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("VAE training history")
    plt.legend()
    savefig(
        os.path.join(outdir, "VAE_training_history.pdf"),
        width=7,
        height=4
    )

    ########################################################
    # Extract latent embedding and reconstruction
    ########################################################

    vae.eval()

    with torch.no_grad():
        X_all = torch.tensor(X_scaled, dtype=torch.float32).to(device)
        z_mean, z_log_var = vae.encode(X_all)
        X_recon = vae.decode(z_mean)

    vae_latent = z_mean.cpu().numpy()
    X_vae_rec = X_recon.cpu().numpy()

    vae_latent_df = pd.DataFrame(
        vae_latent,
        columns=[f"VAE{i + 1}" for i in range(vae_latent_dim)]
    )
    vae_latent_df["Gene"] = gene_ids
    vae_latent_df = vae_latent_df.merge(gene_summary, on="Gene", how="left")

    vae_latent_df.to_csv(
        os.path.join(outdir, "VAE_latent_gene_embedding.csv"),
        index=False
    )

    vae_recon_mse = np.mean((X_scaled - X_vae_rec) ** 2)

    vae_error = pd.DataFrame({
        "Gene": gene_ids,
        "VAE_reconstruction_MSE": np.mean((X_scaled - X_vae_rec) ** 2, axis=1)
    })

    vae_error = (
        vae_error
        .merge(gene_summary, on="Gene", how="left")
        .sort_values("VAE_reconstruction_MSE", ascending=False)
    )

    vae_error.to_csv(
        os.path.join(outdir, "VAE_gene_reconstruction_error.csv"),
        index=False
    )

    ########################################################
    # VAE latent maps
    ########################################################

    plt.figure()
    for space in Spacepoints:
        sub = vae_latent_df[vae_latent_df["strongest_defect_space"] == space]
        plt.scatter(sub["VAE1"], sub["VAE2"], s=10, alpha=0.75, label=space)

    plt.xlabel("VAE latent dimension 1")
    plt.ylabel("VAE latent dimension 2")
    plt.title(
        "VAE latent map of spatiotemporal fitness phenotypes\n"
        "Question: do genes form nonlinear dynamic-fitness manifolds?"
    )
    plt.legend(title="Strongest defect space", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
    savefig(
        os.path.join(outdir, "VAE_latent_map_by_strongest_defect_space.pdf"),
        width=8,
        height=5
    )

    plt.figure()
    for time in Full_Timepoints:
        sub = vae_latent_df[vae_latent_df["strongest_defect_time"] == time]
        plt.scatter(sub["VAE1"], sub["VAE2"], s=10, alpha=0.75, label=time)

    plt.xlabel("VAE latent dimension 1")
    plt.ylabel("VAE latent dimension 2")
    plt.title(
        "VAE latent map of spatiotemporal fitness phenotypes\n"
        "Question: do early, middle, and late fitness defects occupy different latent regions?"
    )
    plt.legend(title="Strongest defect time", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    savefig(
        os.path.join(outdir, "VAE_latent_map_by_strongest_defect_time.pdf"),
        width=8,
        height=5
    )

    ########################################################
    # Observed vs reconstructed heatmaps
    ########################################################

    X_vae_rec_df = pd.DataFrame(
        X_vae_rec,
        index=gene_ids,
        columns=X_scaled_df.columns
    )

    plot_vae_only_reconstruction_examples = False
    if plot_vae_only_reconstruction_examples:
        example_genes = vae_error.head(6)["Gene"].to_list()
        obs_long = matrix_to_long(X_scaled_df.loc[example_genes], value_name="value")
        obs_long["Type"] = "Observed"
        rec_long = matrix_to_long(X_vae_rec_df.loc[example_genes], value_name="value")
        rec_long["Type"] = "VAE reconstructed"
        vae_compare_long = pd.concat([obs_long, rec_long], axis=0)
        plot_gene_matrix_heatmap(
            vae_compare_long,
            title="Observed vs VAE-reconstructed fitness landscapes",
            subtitle="Question: can the model denoise or reconstruct full 5 x 12 profiles?",
            outfile=os.path.join(outdir, "VAE_observed_vs_reconstructed_example_genes.pdf")
        )

        representative_genes = vae_error.tail(6)["Gene"].to_list()
        representative_obs_long = matrix_to_long(X_scaled_df.loc[representative_genes], value_name="value")
        representative_obs_long["Type"] = "Observed"
        representative_rec_long = matrix_to_long(X_vae_rec_df.loc[representative_genes], value_name="value")
        representative_rec_long["Type"] = "VAE reconstructed"
        representative_compare_long = pd.concat([representative_obs_long, representative_rec_long], axis=0)
        plot_gene_matrix_heatmap(
            representative_compare_long,
            title="Observed vs VAE-reconstructed representative fitness landscapes",
            subtitle="Lowest VAE reconstruction error = most typical profiles captured by the model",
            outfile=os.path.join(outdir, "VAE_observed_vs_reconstructed_representative_genes.pdf")
        )

else:
    print("VAE skipped because torch is not installed or run_vae=False.")


Using up to 4 CPU cores for explicit parallel work; BLAS threads per worker = 1.
Gene names loaded from annotation: 3222
Raw data head:
           Gene Time Space    logFC  timeNumber  Time_idx  Space_idx
0  N900_RS00015   1h    st  2.40810           1         1          1
1  N900_RS00025   1h    st -0.17281           1         1          1
2  N900_RS00040   1h    st -0.42050           1         1          1
3  N900_RS00045   1h    st  0.58037           1         1          1
4  N900_RS00050   1h    st -0.43105           1         1          1
Raw data shape: (191640, 7)
Cleaned data shape: (191640, 5)
           Gene Time Space  Feature    logFC
0  N900_RS00015  12h   SI1  12h_SI1  2.53740
1  N900_RS00015  12h   SI2  12h_SI2  2.11260
2  N900_RS00015  12h   SI3  12h_SI3  1.61150
3  N900_RS00015  12h   SI4  12h_SI4  0.29605
4  N900_RS00015  12h   SI5  12h_SI5 -0.75990
Number of genes with complete profiles: 3194
Number of features: 60
Using device for VAE: cpu
Epoch 001 | train loss: 58

In [2]:
############################################################
# Follow-up analysis
# 1. Quantify PCA, Tensor, and VAE model performance
# 2. Analyze and visualize nonlinear dynamic patterns learned by VAE
#
# Run this chunk after the main Module_driver code has finished.
############################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


############################################################
# Helper functions
############################################################

def reconstruction_metrics(observed, reconstructed, model_name):
    observed = np.asarray(observed, dtype=float)
    reconstructed = np.asarray(reconstructed, dtype=float)

    residual = observed - reconstructed
    mse = np.mean(residual ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(residual))

    ss_res = np.sum(residual ** 2)
    ss_tot = np.sum((observed - np.mean(observed)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return {
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2_reconstruction": r2
    }


def plot_latent_landscape_heatmap(decoded_vector, title, outfile):
    decoded_df = pd.DataFrame(
        decoded_vector.reshape(1, -1),
        index=["Decoded"],
        columns=X_scaled_df.columns
    )

    decoded_long = matrix_to_long(decoded_df, value_name="value")
    decoded_long["Type"] = "Decoded landscape"

    plot_gene_matrix_heatmap(
        decoded_long,
        title=title,
        subtitle="Decoded from one location in VAE latent space",
        outfile=outfile
    )


############################################################
# 1. Quantify three-model performance
############################################################

model_performance = []

# PCA reconstruction performance.
# Use PCA-5 for a fair reconstruction comparison with CP-rank-5.
pca_n_components_for_reconstruction = min(5, pca_scores_full.shape[1])

pca_scores_for_reconstruction = pca_scores_full[:, :pca_n_components_for_reconstruction]
pca_components_for_reconstruction = pca.components_[:pca_n_components_for_reconstruction, :]
pca_reconstructed = (
    pca_scores_for_reconstruction @ pca_components_for_reconstruction
    + pca.mean_
)

model_performance.append(
    reconstruction_metrics(
        X_scaled,
        pca_reconstructed,
        f"PCA_{pca_n_components_for_reconstruction}_components"
    )
)

# Tensor reconstruction performance.
if "tensor_recon_matrix" in globals() and np.isfinite(tensor_recon_mse):
    model_performance.append(
        reconstruction_metrics(
            X_scaled,
            tensor_recon_matrix,
            f"Tensor_CP_rank_{tensor_rank}"
        )
    )

# VAE reconstruction performance.
if "X_vae_rec" in globals() and np.isfinite(vae_recon_mse):
    model_performance.append(
        reconstruction_metrics(
            X_scaled,
            X_vae_rec,
            f"VAE_latent_dim_{vae_latent_dim}"
        )
    )

model_performance_df = pd.DataFrame(model_performance)
model_performance_df.to_csv(
    os.path.join(outdir, "Model_performance_reconstruction_metrics.csv"),
    index=False
)

print(model_performance_df)


############################################################
# 2. Can VAE reveal nonlinear dynamic patterns?
############################################################
# Idea:
# - PCA/Tensor impose more linear or multilinear structure.
# - VAE can learn a curved latent manifold.
# - Analyze this by clustering latent coordinates, linking clusters to
#   time/space defect summaries, and decoding latent-space locations back into
#   5 x 12 fitness landscapes.
############################################################

if "vae_latent_df" in globals() and "X_vae_rec" in globals():

    latent_cols = [f"VAE{i + 1}" for i in range(vae_latent_dim)]
    latent_values = vae_latent_df[latent_cols].values

    ########################################################
    # 2.1 Choose latent clusters and quantify separation
    ########################################################

    cluster_records = []
    candidate_k = range(2, 9)

    for k in candidate_k:
        km = KMeans(n_clusters=k, random_state=vae_random_seed, n_init=20)
        labels = km.fit_predict(latent_values)
        sil = silhouette_score(latent_values, labels)
        cluster_records.append({
            "k": k,
            "silhouette_score": sil,
            "inertia": km.inertia_
        })

    vae_cluster_selection = pd.DataFrame(cluster_records)
    best_k = int(
        vae_cluster_selection
        .sort_values(["silhouette_score", "k"], ascending=[False, True])
        .iloc[0]["k"]
    )

    vae_cluster_selection.to_csv(
        os.path.join(outdir, "VAE_latent_cluster_selection.csv"),
        index=False
    )

    plt.figure(figsize=(6, 4))
    plt.plot(
        vae_cluster_selection["k"],
        vae_cluster_selection["silhouette_score"],
        marker="o"
    )
    plt.xlabel("Number of latent clusters")
    plt.ylabel("Silhouette score")
    plt.title("VAE latent cluster selection")
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, "VAE_latent_cluster_selection_silhouette.pdf"),
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )
    plt.close()

    ########################################################
    # 2.2 Visualize nonlinear latent groups
    ########################################################

    kmeans = KMeans(n_clusters=best_k, random_state=vae_random_seed, n_init=20)
    vae_latent_df = vae_latent_df.copy()

    if (
        "VAE_reconstruction_MSE" not in vae_latent_df.columns
        and "vae_error" in globals()
        and "VAE_reconstruction_MSE" in vae_error.columns
    ):
        vae_latent_df = vae_latent_df.merge(
            vae_error[["Gene", "VAE_reconstruction_MSE"]],
            on="Gene",
            how="left"
        )

    vae_latent_df["VAE_cluster"] = kmeans.fit_predict(latent_values).astype(str)

    vae_latent_df.to_csv(
        os.path.join(outdir, "VAE_latent_gene_embedding_with_clusters.csv"),
        index=False
    )

    plt.figure(figsize=(7, 5))
    for cluster in sorted(vae_latent_df["VAE_cluster"].unique()):
        sub = vae_latent_df[vae_latent_df["VAE_cluster"] == cluster]
        plt.scatter(
            sub["VAE1"],
            sub["VAE2"],
            s=14,
            alpha=0.8,
            label=f"Cluster {cluster}"
        )

    centers = kmeans.cluster_centers_
    plt.scatter(
        centers[:, 0],
        centers[:, 1],
        s=120,
        marker="x",
        color="black",
        label="Cluster centers"
    )
    plt.xlabel("VAE latent dimension 1")
    plt.ylabel("VAE latent dimension 2")
    plt.title("VAE nonlinear latent programs")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, "VAE_latent_clusters_nonlinear_programs.pdf"),
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )
    plt.close()

    ########################################################
    # 2.3 Summarize each nonlinear cluster by time and GI space
    ########################################################

    cluster_summary = (
        vae_latent_df
        .groupby("VAE_cluster")
        .agg(
            n_genes=("Gene", "count"),
            mean_reconstruction_MSE=("VAE_reconstruction_MSE", "mean")
            if "VAE_reconstruction_MSE" in vae_latent_df.columns
            else ("Gene", "count"),
            strongest_defect_time_mode=("strongest_defect_time", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
            strongest_defect_space_mode=("strongest_defect_space", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
            mean_fitness_mean=("mean_fitness", "mean"),
            min_fitness_mean=("min_fitness", "mean"),
            dynamic_range_mean=("dynamic_range", "mean")
        )
        .reset_index()
    )

    cluster_summary.to_csv(
        os.path.join(outdir, "VAE_latent_cluster_summary.csv"),
        index=False
    )

    time_cluster = pd.crosstab(
        vae_latent_df["VAE_cluster"],
        vae_latent_df["strongest_defect_time"],
        normalize="index"
    ).reindex(columns=Full_Timepoints)

    space_cluster = pd.crosstab(
        vae_latent_df["VAE_cluster"],
        vae_latent_df["strongest_defect_space"],
        normalize="index"
    ).reindex(columns=Spacepoints)

    time_cluster.to_csv(
        os.path.join(outdir, "VAE_cluster_by_strongest_defect_time_fraction.csv")
    )
    space_cluster.to_csv(
        os.path.join(outdir, "VAE_cluster_by_strongest_defect_space_fraction.csv")
    )

    plt.figure(figsize=(7, 3.5))
    plt.imshow(time_cluster.values, aspect="auto", vmin=0, vmax=np.nanmax(time_cluster.values))
    plt.xticks(range(len(Full_Timepoints)), Full_Timepoints)
    plt.yticks(range(len(time_cluster.index)), time_cluster.index)
    plt.xlabel("Strongest defect time")
    plt.ylabel("VAE cluster")
    plt.title("VAE nonlinear programs by defect timing")
    plt.colorbar(label="Fraction of genes in cluster")
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, "VAE_cluster_defect_time_heatmap.pdf"),
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )
    plt.close()

    plt.figure(figsize=(9, 3.5))
    plt.imshow(space_cluster.values, aspect="auto", vmin=0, vmax=np.nanmax(space_cluster.values))
    plt.xticks(range(len(Spacepoints)), Spacepoints, rotation=45, ha="right")
    plt.yticks(range(len(space_cluster.index)), space_cluster.index)
    plt.xlabel("Strongest defect GI location")
    plt.ylabel("VAE cluster")
    plt.title("VAE nonlinear programs by GI location")
    plt.colorbar(label="Fraction of genes in cluster")
    plt.tight_layout()
    plt.savefig(
        os.path.join(outdir, "VAE_cluster_defect_space_heatmap.pdf"),
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )
    plt.close()

    ########################################################
    # 2.4 Decode each cluster center into a 5 x 12 pattern
    ########################################################

    if "vae" in globals() and "torch" in globals() and "device" in globals():
        vae.eval()

        with torch.no_grad():
            centers_tensor = torch.tensor(centers, dtype=torch.float32).to(device)
            decoded_centers = vae.decode(centers_tensor).cpu().numpy()

        decoded_center_df = pd.DataFrame(
            decoded_centers,
            index=[f"Cluster_{i}" for i in range(best_k)],
            columns=X_scaled_df.columns
        )

        decoded_center_df.to_csv(
            os.path.join(outdir, "VAE_decoded_cluster_center_landscapes.csv")
        )

        decoded_long = matrix_to_long(decoded_center_df, value_name="value")
        decoded_long["Gene"] = decoded_long["Gene"].astype(str)
        decoded_long["Type"] = "Decoded cluster center"

        plot_gene_matrix_heatmap(
            decoded_long,
            title="Decoded VAE nonlinear program landscapes",
            subtitle="Each row is the decoded 5 x 12 fitness pattern at one latent cluster center",
            outfile=os.path.join(outdir, "VAE_decoded_cluster_center_landscapes.pdf")
        )

    print("Saved VAE nonlinear dynamic pattern analysis outputs.")

else:
    print("VAE follow-up skipped because VAE outputs are not available.")


              Model       MSE      RMSE       MAE  R2_reconstruction
0  PCA_5_components  0.478728  0.691901  0.485216           0.521272
1  Tensor_CP_rank_5  0.511345  0.715084  0.500837           0.488655
2  VAE_latent_dim_4  0.484570  0.696111  0.481638           0.515430
Saved VAE nonlinear dynamic pattern analysis outputs.


In [3]:
############################################################
# Follow-up analysis
# PCA linear dynamic pattern programs
#
# Run this chunk after the main Module_driver code has finished.
############################################################

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


############################################################
# User parameters
############################################################

n_pca_programs_to_plot = min(6, pca.components_.shape[0])
n_top_genes_per_pc = 25


############################################################
# 1. PCA component loadings as 5 x 12 linear programs
############################################################

pca_program_records = []

for pc_idx in range(n_pca_programs_to_plot):
    pc_name = f"PC{pc_idx + 1}"

    for feature, loading in zip(X_scaled_df.columns, pca.components_[pc_idx]):
        time, space = parse_feature(feature)
        pca_program_records.append({
            "PC": pc_name,
            "Feature": feature,
            "Time": time,
            "Space": space,
            "Loading": loading,
            "Abs_loading": abs(loading),
            "Variance_explained": pca.explained_variance_ratio_[pc_idx]
        })

pca_program_long = pd.DataFrame(pca_program_records)
pca_program_long["Time"] = pd.Categorical(
    pca_program_long["Time"],
    categories=Full_Timepoints,
    ordered=True
)
pca_program_long["Space"] = pd.Categorical(
    pca_program_long["Space"],
    categories=Spacepoints,
    ordered=True
)

pca_program_long.to_csv(
    os.path.join(outdir, "PCA_linear_dynamic_program_loadings_long.csv"),
    index=False
)


############################################################
# 3. Temporal and spatial summaries for each PCA program
############################################################

pca_time_program = (
    pca_program_long
    .groupby(["PC", "Time"], observed=True)
    .agg(
        mean_loading=("Loading", "mean"),
        mean_abs_loading=("Abs_loading", "mean")
    )
    .reset_index()
)

pca_space_program = (
    pca_program_long
    .groupby(["PC", "Space"], observed=True)
    .agg(
        mean_loading=("Loading", "mean"),
        mean_abs_loading=("Abs_loading", "mean")
    )
    .reset_index()
)

pca_time_program.to_csv(
    os.path.join(outdir, "PCA_linear_program_time_summary.csv"),
    index=False
)

pca_space_program.to_csv(
    os.path.join(outdir, "PCA_linear_program_space_summary.csv"),
    index=False
)

plt.figure(figsize=(8, 4.5))
for pc_idx in range(n_pca_programs_to_plot):
    pc_name = f"PC{pc_idx + 1}"
    sub = pca_time_program[pca_time_program["PC"] == pc_name].copy()
    sub["Time"] = pd.Categorical(sub["Time"], categories=Full_Timepoints, ordered=True)
    sub = sub.sort_values("Time")
    plt.plot(
        sub["Time"].astype(str),
        sub["mean_abs_loading"],
        marker="o",
        label=pc_name
    )

plt.xlabel("Time")
plt.ylabel("Mean absolute PCA loading")
plt.title("Temporal strength of PCA linear programs")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(
    os.path.join(outdir, "PCA_linear_program_temporal_strength.pdf"),
    dpi=300,
    bbox_inches="tight",
    transparent=True
)
plt.close()

plt.figure(figsize=(9, 4.5))
for pc_idx in range(n_pca_programs_to_plot):
    pc_name = f"PC{pc_idx + 1}"
    sub = pca_space_program[pca_space_program["PC"] == pc_name].copy()
    sub["Space"] = pd.Categorical(sub["Space"], categories=Spacepoints, ordered=True)
    sub = sub.sort_values("Space")
    plt.plot(
        sub["Space"].astype(str),
        sub["mean_abs_loading"],
        marker="o",
        label=pc_name
    )

plt.xlabel("GI location")
plt.ylabel("Mean absolute PCA loading")
plt.xticks(rotation=45, ha="right")
plt.title("Spatial strength of PCA linear programs")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(
    os.path.join(outdir, "PCA_linear_program_spatial_strength.pdf"),
    dpi=300,
    bbox_inches="tight",
    transparent=True
)
plt.close()


############################################################
# 4. Top genes defining each PCA linear program
############################################################

pca_gene_program_records = []

for pc_idx in range(n_pca_programs_to_plot):
    pc_name = f"PC{pc_idx + 1}"

    score_df = pd.DataFrame({
        "Gene": gene_ids,
        "PC": pc_name,
        "PC_score": pca_scores_full[:, pc_idx]
    })

    if "gene_summary" in globals():
        score_df = score_df.merge(gene_summary, on="Gene", how="left")

    score_df["Abs_PC_score"] = score_df["PC_score"].abs()
    score_df["Direction"] = np.where(score_df["PC_score"] >= 0, "Positive", "Negative")

    top_positive = (
        score_df
        .sort_values("PC_score", ascending=False)
        .head(n_top_genes_per_pc)
    )
    top_negative = (
        score_df
        .sort_values("PC_score", ascending=True)
        .head(n_top_genes_per_pc)
    )

    pca_gene_program_records.append(top_positive)
    pca_gene_program_records.append(top_negative)

pca_top_genes = pd.concat(pca_gene_program_records, axis=0).reset_index(drop=True)

if "Gene_display" not in pca_top_genes.columns:
    pca_top_genes["Gene_display"] = pca_top_genes["Gene"]

pca_top_genes.to_csv(
    os.path.join(outdir, "PCA_top_genes_per_linear_program.csv"),
    index=False
)


############################################################
# 5. Plot top positive and negative genes for each PCA program
############################################################

fig, axes = plt.subplots(n_pca_programs_to_plot, 2, figsize=(12, 3.6 * n_pca_programs_to_plot))

if n_pca_programs_to_plot == 1:
    axes = np.array([axes])

for pc_idx in range(n_pca_programs_to_plot):
    pc_name = f"PC{pc_idx + 1}"

    pos_ax = axes[pc_idx, 0]
    neg_ax = axes[pc_idx, 1]

    pos = (
        pca_top_genes[
            (pca_top_genes["PC"] == pc_name)
            & (pca_top_genes["Direction"] == "Positive")
        ]
        .sort_values("PC_score")
    )

    neg = (
        pca_top_genes[
            (pca_top_genes["PC"] == pc_name)
            & (pca_top_genes["Direction"] == "Negative")
        ]
        .sort_values("PC_score", ascending=False)
    )

    pos_ax.barh(pos["Gene_display"], pos["PC_score"])
    pos_ax.set_title(f"{pc_name} positive genes")
    pos_ax.set_xlabel("PC score")
    pos_ax.tick_params(axis="y", labelsize=7)

    neg_ax.barh(neg["Gene_display"], neg["PC_score"])
    neg_ax.set_title(f"{pc_name} negative genes")
    neg_ax.set_xlabel("PC score")
    neg_ax.tick_params(axis="y", labelsize=7)

fig.suptitle(
    "Genes defining PCA linear dynamic programs\n"
    "Positive and negative scores represent opposite ends of each linear program",
    fontsize=13
)
plt.tight_layout()
plt.savefig(
    os.path.join(outdir, "PCA_top_genes_per_linear_program.pdf"),
    dpi=300,
    bbox_inches="tight",
    transparent=True
)
plt.close()


############################################################
# 6. Representative observed landscapes for PCA extremes
############################################################

plot_pca_representative_landscapes = False

if plot_pca_representative_landscapes:
    for pc_idx in range(n_pca_programs_to_plot):
        pc_name = f"PC{pc_idx + 1}"

        pc_scores = pd.DataFrame({
            "Gene": gene_ids,
            "PC_score": pca_scores_full[:, pc_idx]
        })

        positive_examples = (
            pc_scores
            .sort_values("PC_score", ascending=False)
            .head(3)["Gene"]
            .to_list()
        )
        negative_examples = (
            pc_scores
            .sort_values("PC_score", ascending=True)
            .head(3)["Gene"]
            .to_list()
        )

        example_genes = positive_examples + negative_examples

        pc_example_long = matrix_to_long(
            X_scaled_df.loc[example_genes],
            value_name="value"
        )
        pc_example_long["Type"] = [
            "Positive PC extreme" if gene in positive_examples else "Negative PC extreme"
            for gene in pc_example_long["Gene"]
        ]

        plot_gene_matrix_heatmap(
            pc_example_long,
            title=f"{pc_name} representative observed fitness landscapes",
            subtitle="Genes at opposite ends of a PCA program show opposite linear dynamic patterns",
            outfile=os.path.join(outdir, f"{pc_name}_representative_linear_program_gene_landscapes.pdf")
        )


print("Saved PCA linear dynamic program follow-up outputs.")


Saved PCA linear dynamic program follow-up outputs.


In [4]:
# %% Integrated rank-4 PCA / CP / VAE comparison
############################################################
# 10. Integrated rank-4 model comparison and TF enrichment
############################################################
# Adds:
# 1. Decoded PCA linear program landscapes.
# 2. Combined PCA / CP / VAE rank-4 landscape figure.
# 3. Combined top-20 gene contribution figure with putative TF labels.
# 4. Reconstruction ROC curves comparing PCA, CP, and VAE quality.
############################################################

comparison_rank = 4


def load_putative_tf_genes(tf_path):
    if not os.path.exists(tf_path):
        warnings.warn(f"Putative TF file not found: {tf_path}")
        return set(), pd.DataFrame()
    tf_df = pd.read_excel(tf_path)
    if "locus_ID" not in tf_df.columns:
        raise ValueError("putative_transcription_regulators.xlsx must contain a locus_ID column.")
    tf_genes = set(
        tf_df["locus_ID"]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne("")]
        .tolist()
    )
    return tf_genes, tf_df


putative_tf_genes, putative_tf_table = load_putative_tf_genes(putative_tf_file)
print(f"Loaded putative transcription regulators: {len(putative_tf_genes)} unique identifiers")


def add_putative_tf_flag(df, gene_col="Gene"):
    df = df.copy()
    gene_values = df[gene_col].astype(str).str.strip()
    df["is_putative_TF"] = gene_values.isin(putative_tf_genes)
    return df


gene_summary_tf = add_putative_tf_flag(gene_summary, "Gene")


def vector_to_landscape(vector, columns=None):
    if columns is None:
        columns = X_scaled_df.columns
    vector_series = pd.Series(np.asarray(vector, dtype=float), index=columns)
    ordered = vector_series.reindex(feature_order)
    return ordered.values.reshape(len(Full_Timepoints), len(Spacepoints))


def save_landscape_grid(programs, outfile, title, value_label="Decoded value / loading", scale_mode="global"):
    if not programs:
        return

    def display_matrix(matrix):
        values = np.asarray(matrix, dtype=float)
        if scale_mode == "per_panel_zscore":
            center = np.nanmedian(values)
            scale = np.nanstd(values)
            if not np.isfinite(scale) or scale == 0:
                scale = 1.0
            return (values - center) / scale
        return values

    nrows = len(programs)
    ncols = len(programs[0])
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(3.6 * ncols, 2.15 * nrows),
        squeeze=False
    )
    display_values = np.concatenate([
        display_matrix(item["matrix"]).ravel()
        for row in programs
        for item in row
        if item is not None
    ])
    vmax = np.nanquantile(np.abs(display_values), 0.98)
    if scale_mode == "per_panel_zscore":
        vmax = min(max(vmax, 1.5), 3.0)
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    vmin = -vmax
    im = None
    for row_idx, row in enumerate(programs):
        for col_idx, item in enumerate(row):
            ax = axes[row_idx, col_idx]
            if item is None:
                ax.axis("off")
                continue
            im = ax.imshow(display_matrix(item["matrix"]), aspect="auto", cmap="coolwarm", vmin=vmin, vmax=vmax)
            ax.set_title(item["title"], fontsize=9)
            ax.set_xticks(range(len(Spacepoints)))
            ax.set_xticklabels(Spacepoints, rotation=45, ha="right", fontsize=7)
            ax.set_yticks(range(len(Full_Timepoints)))
            ax.set_yticklabels(Full_Timepoints, fontsize=7)
    fig.suptitle(title, fontsize=13)
    if im is not None:
        fig.subplots_adjust(hspace=0.72, wspace=0.30, top=0.92, bottom=0.08, right=0.86)
        cbar_ax = fig.add_axes([0.89, 0.18, 0.018, 0.64])
        fig.colorbar(im, cax=cbar_ax, label=value_label)
    else:
        fig.subplots_adjust(hspace=0.72, wspace=0.30, top=0.92, bottom=0.08, right=0.90)
    plt.savefig(outfile, dpi=300, bbox_inches="tight", transparent=True)
    plt.close()


############################################################
# Observed vs PCA / CP / VAE reconstruction for VAE-selected genes
############################################################

reconstruction_compare_models = {}
reconstruction_compare_models["PCA reconstructed"] = (
    pca_scores_full[:, :comparison_rank] @ pca.components_[:comparison_rank, :]
    + pca.mean_
)
if "tensor_recon_matrix" in globals():
    reconstruction_compare_models["CP reconstructed"] = tensor_recon_matrix
if "X_vae_rec" in globals():
    reconstruction_compare_models["VAE reconstructed"] = X_vae_rec

if "X_vae_rec" in globals() and len(reconstruction_compare_models) >= 2:
    reconstruction_error_df = pd.DataFrame({"Gene": gene_ids})
    for model_label, reconstructed in reconstruction_compare_models.items():
        clean_label = model_label.replace(" reconstructed", "").replace(" ", "_")
        reconstruction_error_df[f"{clean_label}_MSE"] = np.mean(
            (X_scaled - np.asarray(reconstructed, dtype=float)) ** 2,
            axis=1
        )
    reconstruction_error_df = reconstruction_error_df.merge(gene_summary, on="Gene", how="left")
    reconstruction_error_df.to_csv(
        os.path.join(outdir, "PCA_CP_VAE_gene_reconstruction_errors.csv"),
        index=False
    )

    def plot_model_reconstruction_gene_set(gene_list, outfile, title, subtitle):
        pieces = []
        obs = matrix_to_long(X_scaled_df.loc[gene_list], value_name="value")
        obs["Type"] = "Observed"
        pieces.append(obs)
        for model_label, reconstructed in reconstruction_compare_models.items():
            rec_df = pd.DataFrame(reconstructed, index=gene_ids, columns=X_scaled_df.columns)
            rec_long = matrix_to_long(rec_df.loc[gene_list], value_name="value")
            rec_long["Type"] = model_label
            pieces.append(rec_long)
        compare_long = pd.concat(pieces, axis=0)
        plot_gene_matrix_heatmap(compare_long, title=title, subtitle=subtitle, outfile=outfile)

    n_gene_examples = 6
    good_genes = (
        reconstruction_error_df
        .sort_values("VAE_MSE", ascending=True)
        .head(n_gene_examples)["Gene"]
        .to_list()
    )
    poor_genes = (
        reconstruction_error_df
        .sort_values("VAE_MSE", ascending=False)
        .head(n_gene_examples)["Gene"]
        .to_list()
    )

    if {"PCA_MSE", "CP_MSE", "VAE_MSE"}.issubset(reconstruction_error_df.columns):
        vae_specific_df = reconstruction_error_df.copy()
        vae_specific_df["VAE_advantage"] = (
            vae_specific_df[["PCA_MSE", "CP_MSE"]].min(axis=1)
            - vae_specific_df["VAE_MSE"]
        )
        vae_specific_genes = (
            vae_specific_df
            .sort_values(["VAE_advantage", "VAE_MSE"], ascending=[False, True])
            .head(n_gene_examples)["Gene"]
            .to_list()
        )
    else:
        vae_specific_genes = good_genes

    plot_model_reconstruction_gene_set(
        good_genes,
        os.path.join(outdir, "PCA_CP_VAE_reconstruction_good_VAE_selected_genes.pdf"),
        "Observed vs reconstructed landscapes for well-predicted genes",
        "Genes selected by lowest VAE reconstruction error"
    )
    plot_model_reconstruction_gene_set(
        poor_genes,
        os.path.join(outdir, "PCA_CP_VAE_reconstruction_poor_VAE_selected_genes.pdf"),
        "Observed vs reconstructed landscapes for poorly predicted genes",
        "Genes selected by highest VAE reconstruction error"
    )
    plot_model_reconstruction_gene_set(
        vae_specific_genes,
        os.path.join(outdir, "PCA_CP_VAE_reconstruction_VAE_specific_good_genes.pdf"),
        "Observed vs reconstructed landscapes for VAE-specific good predictions",
        "Genes where VAE has the largest reconstruction-error advantage over PCA and CP"
    )

print("Saved integrated PCA / CP / VAE rank-4 comparison outputs.")


Loaded putative transcription regulators: 211 unique identifiers
Saved integrated PCA / CP / VAE rank-4 comparison outputs.


In [5]:
# %% CP-structured decoder / tensor-factorized VAE
############################################################
# 11. Fourth model: CP-structured decoder VAE
############################################################
# Model idea:
# - Encoder is VAE-like and maps each gene profile to latent program weights.
# - Decoder is CP/tensor-factorized: each latent dimension decodes through
#   a learned Time factor x Space factor module.
# - This keeps VAE nonlinear gene embedding while making hidden fitness
#   programs directly interpretable as CP-style time-space modules.
############################################################
    
cpvae_rank = 6
cpvae_epochs = 180
cpvae_scan_epochs = 20
cpvae_patience = 15
cpvae_learning_rate = vae_learning_rate if "vae_learning_rate" in globals() else 1e-3
cpvae_kl_weight = 0.001
cpvae_factor_orthogonality_weight = 0.015
cpvae_latent_decorrelation_weight = 0.01
cpvae_factor_sparsity_weight = 0.00005
cpvae_residual_weight = 0.25
cpvae_residual_penalty_weight = 0.02
cpvae_warmup_epochs = 40
cpvae_signal_weight_alpha = 0.6
cpvae_validation_signal_mix = 0.25
cpvae_cp_branch_loss_weight = 0.35
cpvae_landscape_epochs = 90
cpvae_landscape_patience = 8
cpvae_random_seed = vae_random_seed if "vae_random_seed" in globals() else 1
    
if run_vae and HAS_TORCH and all(f in X_scaled_df.columns for f in feature_order):
    cpvae_device = device if "device" in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    class CPStructuredVAE(nn.Module):
        def __init__(self, input_dim, n_time, n_space, latent_dim=4, hidden_1=64, hidden_2=32):
            super().__init__()
            self.n_time = n_time
            self.n_space = n_space
            self.latent_dim = latent_dim
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden_1),
                nn.ReLU(),
                nn.Linear(hidden_1, hidden_2),
                nn.ReLU()
            )
            self.z_mean = nn.Linear(hidden_2, latent_dim)
            self.z_log_var = nn.Linear(hidden_2, latent_dim)
            self.time_factor = nn.Parameter(torch.randn(n_time, latent_dim) * 0.05)
            self.space_factor = nn.Parameter(torch.randn(n_space, latent_dim) * 0.05)
            self.feature_bias = nn.Parameter(torch.zeros(n_time * n_space))
            self.residual_decoder = nn.Sequential(
                nn.Linear(latent_dim, hidden_2),
                nn.ReLU(),
                nn.Linear(hidden_2, n_time * n_space)
            )
            self.residual_scale = nn.Parameter(torch.tensor(float(cpvae_residual_weight)))

        def encode(self, x):
            h = self.encoder(x)
            return self.z_mean(h), self.z_log_var(h)

        def reparameterize(self, mean, log_var):
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mean + eps * std

        def cp_decode(self, z):
            # basis: latent_dim x time x space -> latent_dim x feature
            basis = torch.einsum("tr,sr->rts", self.time_factor, self.space_factor)
            basis = basis.reshape(self.latent_dim, self.n_time * self.n_space)
            return z @ basis + self.feature_bias

        def decode(self, z, include_residual=True):
            cp_part = self.cp_decode(z)
            if not include_residual:
                return cp_part
            residual = torch.tanh(self.residual_scale) * self.residual_decoder(z)
            return cp_part + residual

        def forward(self, x):
            mean, log_var = self.encode(x)
            z = self.reparameterize(mean, log_var)
            recon = self.decode(z)
            return recon, mean, log_var

    def off_diagonal_mean_square(matrix):
        if matrix.shape[0] <= 1 or matrix.shape[1] <= 1:
            return torch.tensor(0.0, device=matrix.device)
        gram = matrix.T @ matrix
        off_diag = gram - torch.diag(torch.diag(gram))
        return torch.mean(off_diag.pow(2))

    def cpvae_interpretability_regularization(model, mean):
        time_norm = nn.functional.normalize(model.time_factor, dim=0)
        space_norm = nn.functional.normalize(model.space_factor, dim=0)
        factor_orthogonality = (
            off_diagonal_mean_square(time_norm)
            + off_diagonal_mean_square(space_norm)
        )

        centered_mean = mean - mean.mean(dim=0, keepdim=True)
        latent_scale = centered_mean.std(dim=0, keepdim=True).clamp_min(1e-6)
        latent_norm = centered_mean / latent_scale
        latent_decorrelation = off_diagonal_mean_square(latent_norm / np.sqrt(max(1, latent_norm.shape[0] - 1)))

        factor_sparsity = (
            torch.mean(torch.abs(model.time_factor))
            + torch.mean(torch.abs(model.space_factor))
        )
        residual_penalty = torch.mean(torch.abs(torch.tanh(model.residual_scale) * model.residual_decoder(mean)))
        return (
            cpvae_factor_orthogonality_weight * factor_orthogonality
            + cpvae_latent_decorrelation_weight * latent_decorrelation
            + cpvae_factor_sparsity_weight * factor_sparsity
            + cpvae_residual_penalty_weight * residual_penalty
        )

    def cpvae_weighted_reconstruction_loss(recon_x, x):
        signal_weight = 1.0 + cpvae_signal_weight_alpha * torch.abs(x)
        return torch.sum(signal_weight * (recon_x - x).pow(2))

    def cpvae_loss_function(recon_x, x, mean, log_var, model, kl_weight=0.01, regularization_scale=1.0):
        recon_loss = cpvae_weighted_reconstruction_loss(recon_x, x)
        cp_only_recon = model.decode(mean, include_residual=False)
        cp_branch_loss = cpvae_weighted_reconstruction_loss(cp_only_recon, x)
        kl_loss = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
        interpretability_loss = cpvae_interpretability_regularization(model, mean) * x.shape[0]
        cp_branch_scale = 0.2 + 0.8 * regularization_scale
        return (
            recon_loss
            + cpvae_cp_branch_loss_weight * cp_branch_scale * cp_branch_loss
            + regularization_scale * (kl_weight * kl_loss + interpretability_loss)
        )

    def train_cp_structured_vae(latent_dim, epochs=80, patience=8):
        torch.manual_seed(cpvae_random_seed + latent_dim)
        np.random.seed(cpvae_random_seed + latent_dim)
        model = CPStructuredVAE(
            input_dim=X_scaled.shape[1],
            n_time=len(Full_Timepoints),
            n_space=len(Spacepoints),
            latent_dim=latent_dim,
            hidden_1=max(128, vae_hidden_1),
            hidden_2=max(64, vae_hidden_2)
        ).to(cpvae_device)
        if "time_factor" in globals() and "space_factor" in globals() and latent_dim <= time_factor.shape[1]:
            with torch.no_grad():
                model.time_factor.copy_(torch.tensor(time_factor[:, :latent_dim], dtype=torch.float32, device=cpvae_device))
                model.space_factor.copy_(torch.tensor(space_factor[:, :latent_dim], dtype=torch.float32, device=cpvae_device))
        optimizer_local = optim.Adam(model.parameters(), lr=cpvae_learning_rate)
        best_state = None
        best_val_loss = np.inf
        epochs_without_improvement = 0
        history_rows = []

        for epoch in range(1, epochs + 1):
            model.train()
            train_loss = 0.0
            for batch in train_loader:
                x_batch = batch[0].to(cpvae_device)
                optimizer_local.zero_grad()
                recon_batch, mean, log_var = model(x_batch)
                regularization_scale = min(1.0, max(0.0, (epoch - cpvae_warmup_epochs) / max(1, epochs - cpvae_warmup_epochs)))
                loss = cpvae_loss_function(
                    recon_batch,
                    x_batch,
                    mean,
                    log_var,
                    model,
                    cpvae_kl_weight,
                    regularization_scale=regularization_scale
                )
                loss.backward()
                optimizer_local.step()
                train_loss += loss.item()
            train_loss /= n_train

            model.eval()
            val_loss = 0.0
            val_mse_loss = 0.0
            val_weighted_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    x_batch = batch[0].to(cpvae_device)
                    recon_batch, mean, log_var = model(x_batch)
                    val_loss += cpvae_loss_function(
                        recon_batch,
                        x_batch,
                        mean,
                        log_var,
                        model,
                        cpvae_kl_weight,
                        regularization_scale=regularization_scale
                    ).item()
                    val_mse_loss += nn.functional.mse_loss(recon_batch, x_batch, reduction="sum").item()
                    val_weighted_loss += cpvae_weighted_reconstruction_loss(recon_batch, x_batch).item()
            val_loss /= n_val
            val_mse_loss /= n_val
            val_weighted_loss /= n_val
            val_recon_loss = (
                (1 - cpvae_validation_signal_mix) * val_mse_loss
                + cpvae_validation_signal_mix * val_weighted_loss
            )
            history_rows.append({"latent_dim": latent_dim, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_mse_loss": val_mse_loss, "val_weighted_loss": val_weighted_loss, "val_recon_loss": val_recon_loss, "regularization_scale": regularization_scale})

            if val_recon_loss < best_val_loss - 1e-6:
                best_val_loss = val_recon_loss
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

        if best_state is not None:
            model.load_state_dict(best_state)
            model.to(cpvae_device)
        model.eval()
        with torch.no_grad():
            X_all_cpvae = torch.tensor(X_scaled, dtype=torch.float32).to(cpvae_device)
            z_mean_cpvae, z_log_var_cpvae = model.encode(X_all_cpvae)
            X_cpvae_rec_local = model.decode(z_mean_cpvae, include_residual=True).cpu().numpy()
            X_cpvae_cp_only_local = model.decode(z_mean_cpvae, include_residual=False).cpu().numpy()
        return model, X_cpvae_rec_local, X_cpvae_cp_only_local, z_mean_cpvae.cpu().numpy(), pd.DataFrame(history_rows)

    cpvae_model, X_cpvae_rec, X_cpvae_cp_only_rec, cpvae_latent, cpvae_history = train_cp_structured_vae(
        cpvae_rank,
        epochs=cpvae_epochs,
        patience=cpvae_patience
    )
    cpvae_history.to_csv(
        os.path.join(outdir, "CPVAE_rank6_training_history.csv"),
        index=False
    )

    cpvae_latent_df = pd.DataFrame(
        cpvae_latent,
        columns=[f"CPVAE{i + 1}" for i in range(cpvae_rank)]
    )
    cpvae_latent_df["Gene"] = gene_ids
    cpvae_latent_df = cpvae_latent_df.merge(gene_summary, on="Gene", how="left")
    cpvae_latent_df.to_csv(
        os.path.join(outdir, "CPVAE_rank6_latent_gene_embedding.csv"),
        index=False
    )
    cpvae_branch_metrics = pd.DataFrame([
        reconstruction_metrics(X_scaled, X_cpvae_cp_only_rec, "CPVAE_CP_factorized_branch_only"),
        reconstruction_metrics(X_scaled, X_cpvae_rec, "CPVAE_total_with_constrained_residual")
    ])
    cpvae_branch_metrics.to_csv(
        os.path.join(outdir, "CPVAE_rank6_cp_branch_vs_total_reconstruction_metrics.csv"),
        index=False
    )

    cpvae_time_factor = cpvae_model.time_factor.detach().cpu().numpy()
    cpvae_space_factor = cpvae_model.space_factor.detach().cpu().numpy()
    cpvae_time_factor_df = pd.DataFrame(cpvae_time_factor, columns=[f"CPVAE{i + 1}" for i in range(cpvae_rank)])
    cpvae_time_factor_df["Time"] = Full_Timepoints
    cpvae_space_factor_df = pd.DataFrame(cpvae_space_factor, columns=[f"CPVAE{i + 1}" for i in range(cpvae_rank)])
    cpvae_space_factor_df["Space"] = Spacepoints
    cpvae_time_factor_df.to_csv(os.path.join(outdir, "CPVAE_rank6_time_factors.csv"), index=False)
    cpvae_space_factor_df.to_csv(os.path.join(outdir, "CPVAE_rank6_space_factors.csv"), index=False)

    def normalized_entropy(values):
        weights = np.abs(np.asarray(values, dtype=float))
        total = weights.sum()
        if total <= 0 or len(weights) <= 1:
            return 0.0
        prob = weights / total
        prob = prob[prob > 0]
        return float(-np.sum(prob * np.log(prob)) / np.log(len(weights)))

    cpvae_interpretability_rows = []
    for idx in range(cpvae_rank):
        time_values = cpvae_time_factor[:, idx]
        space_values = cpvae_space_factor[:, idx]
        program_matrix = np.outer(time_values, space_values)
        peak_flat_idx = int(np.nanargmax(np.abs(program_matrix)))
        peak_time_idx, peak_space_idx = np.unravel_index(peak_flat_idx, program_matrix.shape)
        cpvae_interpretability_rows.append({
            "Program": f"CPVAE{idx + 1}",
            "Peak_time": Full_Timepoints[peak_time_idx],
            "Peak_space": Spacepoints[peak_space_idx],
            "Peak_abs_loading": float(np.abs(program_matrix[peak_time_idx, peak_space_idx])),
            "Time_entropy_0_specific_1_diffuse": normalized_entropy(time_values),
            "Space_entropy_0_specific_1_diffuse": normalized_entropy(space_values),
            "Mean_abs_time_loading": float(np.mean(np.abs(time_values))),
            "Mean_abs_space_loading": float(np.mean(np.abs(space_values)))
        })
    cpvae_interpretability_df = pd.DataFrame(cpvae_interpretability_rows)
    cpvae_interpretability_df.to_csv(
        os.path.join(outdir, "CPVAE_rank6_interpretable_program_summary.csv"),
        index=False
    )

    cpvae_time_corr = pd.DataFrame(
        np.corrcoef(cpvae_time_factor.T),
        index=[f"CPVAE{i + 1}" for i in range(cpvae_rank)],
        columns=[f"CPVAE{i + 1}" for i in range(cpvae_rank)]
    )
    cpvae_space_corr = pd.DataFrame(
        np.corrcoef(cpvae_space_factor.T),
        index=[f"CPVAE{i + 1}" for i in range(cpvae_rank)],
        columns=[f"CPVAE{i + 1}" for i in range(cpvae_rank)]
    )
    cpvae_time_corr.to_csv(os.path.join(outdir, "CPVAE_rank6_time_factor_correlation.csv"))
    cpvae_space_corr.to_csv(os.path.join(outdir, "CPVAE_rank6_space_factor_correlation.csv"))

    cpvae_program_records = []
    cpvae_program_rows = []
    for idx in range(cpvae_rank):
        program_name = f"CPVAE{idx + 1}"
        program_matrix = np.outer(cpvae_time_factor[:, idx], cpvae_space_factor[:, idx])
        cpvae_program_rows.append([{"title": program_name, "matrix": program_matrix}])
        for ti, time in enumerate(Full_Timepoints):
            for si, space in enumerate(Spacepoints):
                cpvae_program_records.append({
                    "Model": "CPVAE",
                    "Program": program_name,
                    "Time": time,
                    "Space": space,
                    "Value": program_matrix[ti, si]
                    })
    cpvae_program_df = pd.DataFrame(cpvae_program_records)
    cpvae_program_df.to_csv(os.path.join(outdir, "CPVAE_rank6_program_landscapes_long.csv"), index=False)
    save_landscape_grid(
        cpvae_program_rows,
        os.path.join(outdir, "CPVAE_rank6_program_landscapes.pdf"),
        "CP-structured decoder VAE program landscapes",
        value_label="Tensor-factorized decoder loading"
    )

    cpvae_time_importance = (
        cpvae_program_df
        .assign(Abs_value=lambda df: df["Value"].abs())
        .groupby(["Program", "Time"], observed=True)
        .agg(mean_abs_value=("Abs_value", "mean"), max_abs_value=("Abs_value", "max"))
        .reset_index()
    )
    cpvae_time_importance.to_csv(os.path.join(outdir, "CPVAE_rank6_hidden_program_temporal_importance.csv"), index=False)
    cpvae_time_heatmap = (
        cpvae_time_importance
        .pivot(index="Program", columns="Time", values="mean_abs_value")
        .reindex(columns=Full_Timepoints)
    )
    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    im = ax.imshow(cpvae_time_heatmap.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(Full_Timepoints)))
    ax.set_xticklabels(Full_Timepoints)
    ax.set_yticks(range(len(cpvae_time_heatmap.index)))
    ax.set_yticklabels(cpvae_time_heatmap.index)
    ax.set_xlabel("Time")
    ax.set_ylabel("CPVAE hidden program")
    ax.set_title("When CP-structured VAE hidden programs become important")
    fig.colorbar(im, ax=ax, label="Mean absolute decoder loading")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "CPVAE_rank6_hidden_program_temporal_importance_heatmap.pdf"), dpi=300, bbox_inches="tight", transparent=True)
    plt.close()

    def train_vanilla_vae_for_four_model_rank(rank, epochs=120, patience=10, return_model=False):
        if not (run_vae and HAS_TORCH and "VAE" in globals() and "vae_loss_function" in globals()):
            return None
        torch.manual_seed(vae_random_seed + 100 + rank)
        np.random.seed(vae_random_seed + 100 + rank)
        local_device = device if "device" in globals() else torch.device("cpu")
        model = VAE(
            input_dim=X_scaled.shape[1],
            hidden_1=max(128, vae_hidden_1),
            hidden_2=max(64, vae_hidden_2),
            latent_dim=rank
        ).to(local_device)
        optimizer_local = optim.Adam(model.parameters(), lr=vae_learning_rate)
        best_state = None
        best_val_loss = np.inf
        epochs_without_improvement = 0
        for epoch in range(1, epochs + 1):
            model.train()
            for batch in train_loader:
                x_batch = batch[0].to(local_device)
                optimizer_local.zero_grad()
                recon_batch, mean, log_var = model(x_batch)
                loss = vae_loss_function(recon_batch, x_batch, mean, log_var)
                loss.backward()
                optimizer_local.step()
            model.eval()
            val_recon_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    x_batch = batch[0].to(local_device)
                    recon_batch, mean, log_var = model(x_batch)
                    val_recon_loss += nn.functional.mse_loss(recon_batch, x_batch, reduction="sum").item()
            val_recon_loss /= n_val
            if val_recon_loss < best_val_loss - 1e-6:
                best_val_loss = val_recon_loss
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
        if best_state is not None:
            model.load_state_dict(best_state)
            model.to(local_device)
        model.eval()
        with torch.no_grad():
            X_all_local = torch.tensor(X_scaled, dtype=torch.float32).to(local_device)
            z_mean_local, _ = model.encode(X_all_local)
            X_rec_local = model.decode(z_mean_local).cpu().numpy()
        if return_model:
            return model, X_rec_local, z_mean_local.cpu().numpy()
        return X_rec_local

    def cp_reconstruction_for_four_model_rank(rank, return_factors=False):
        if "tensor_array" not in globals() or "parafac" not in globals() or "cp_to_tensor" not in globals():
            return None
        cp_rank_result = parafac(
            tensor_array,
            rank=rank,
            n_iter_max=80,
            tol=1e-4,
            init="svd",
            random_state=1,
            normalize_factors=False
        )
        cp_rank_tensor = cp_to_tensor(cp_rank_result)
        cp_rank_matrix = cp_rank_tensor.reshape(len(gene_ids), len(Full_Timepoints) * len(Spacepoints))
        cp_rank_df = pd.DataFrame(cp_rank_matrix, index=gene_ids, columns=feature_order)
        cp_rank_rec = cp_rank_df[X_scaled_df.columns].values
        if return_factors:
            weights_rank, factors_rank = cp_rank_result
            return cp_rank_rec, weights_rank, factors_rank
        return cp_rank_rec

    ########################################################
    # Rank 2-10 program distinctness comparison
    ########################################################

    def program_correlation_from_landscape_matrices(matrices, labels):
        valid_vectors = []
        valid_labels = []
        for matrix, label in zip(matrices, labels):
            if matrix is None:
                continue
            vec = np.asarray(matrix, dtype=float).reshape(-1)
            if np.all(~np.isfinite(vec)) or np.nanstd(vec) == 0:
                continue
            valid_vectors.append(vec)
            valid_labels.append(label)
        if len(valid_vectors) < 2:
            return None
        corr = np.corrcoef(np.vstack(valid_vectors))
        return pd.DataFrame(corr, index=valid_labels, columns=valid_labels)

    def decoded_vae_program_matrices(model, latent_dim, latent_values, local_device):
        matrices = []
        labels = []
        model.eval()
        with torch.no_grad():
            z_zero = torch.zeros((1, latent_dim), dtype=torch.float32).to(local_device)
            decoded_zero = model.decode(z_zero).cpu().numpy().reshape(-1)
        for idx in range(latent_dim):
            latent_scale = 1.0
            if latent_values is not None and latent_values.shape[1] > idx:
                latent_scale = np.nanstd(latent_values[:, idx])
                if not np.isfinite(latent_scale) or latent_scale == 0:
                    latent_scale = 1.0
            with torch.no_grad():
                z_axis = torch.zeros((1, latent_dim), dtype=torch.float32).to(local_device)
                z_axis[0, idx] = float(latent_scale)
                decoded_vector = model.decode(z_axis).cpu().numpy().reshape(-1) - decoded_zero
            matrices.append(vector_to_landscape(decoded_vector, X_scaled_df.columns))
            labels.append(f"VAE{idx + 1}")
        return matrices, labels

    rank_distinctness_values = list(range(2, 11))
    rank_distinctness_models = ["PCA", "CP", "VAE", "weighted CPVAE"]
    rank_distinctness_corrs = {}
    rank_distinctness_long_rows = []
    rank_distinctness_summary_rows = []
    rank_model_benchmark_df = pd.DataFrame()
    model_comparison_dir = os.path.join(os.path.dirname(outdir), "model comparison")
    os.makedirs(model_comparison_dir, exist_ok=True)

    # For program distinctness, fit the maximum rank once and display the first N
    # programs for rank 2...10. This avoids retraining two neural models nine times.
    rank_program_max_rank = max(rank_distinctness_values)
    rank_cp_factors = {}
    rank_vae_models = {}
    rank_vae_latents = {}
    rank_cpvae_models = {}
    rank_cpvae_predictions = {}

    pca_max_prediction = pca_scores_full[:, :rank_program_max_rank] @ pca.components_[:rank_program_max_rank, :] + pca.mean_
    cp_max_prediction = None
    cp_max_output = cp_reconstruction_for_four_model_rank(rank_program_max_rank, return_factors=True)
    cp_max_factors = None
    if cp_max_output is not None:
        cp_max_prediction, _, cp_max_factors = cp_max_output
        for rank in rank_distinctness_values:
            rank_cp_factors[rank] = cp_max_factors

    vae_max_result = train_vanilla_vae_for_four_model_rank(
        rank_program_max_rank,
        epochs=cpvae_scan_epochs,
        patience=3,
        return_model=True
    )
    vae_max_model = None
    vae_max_prediction = None
    vae_max_latent = None
    if vae_max_result is not None:
        vae_max_model, vae_max_prediction, vae_max_latent = vae_max_result
        for rank in rank_distinctness_values:
            rank_vae_models[rank] = vae_max_model
            rank_vae_latents[rank] = vae_max_latent

    if rank_program_max_rank == cpvae_rank:
        cpvae_max_model = cpvae_model
        cpvae_max_prediction = X_cpvae_rec
    else:
        cpvae_max_model, cpvae_max_prediction, _, _, _ = train_cp_structured_vae(
            rank_program_max_rank,
            epochs=cpvae_landscape_epochs,
            patience=cpvae_landscape_patience
        )
    for rank in rank_distinctness_values:
        rank_cpvae_models[rank] = cpvae_max_model
        rank_cpvae_predictions[rank] = cpvae_max_prediction

    vae_max_matrices, vae_max_labels = ([], [])
    if vae_max_model is not None:
        vae_max_matrices, vae_max_labels = decoded_vae_program_matrices(
            vae_max_model,
            rank_program_max_rank,
            vae_max_latent,
            device if "device" in globals() else torch.device("cpu")
        )
    cpvae_max_time = cpvae_max_model.time_factor.detach().cpu().numpy()
    cpvae_max_space = cpvae_max_model.space_factor.detach().cpu().numpy()

    for rank in rank_distinctness_values:
        rank_corrs = {}

        pca_matrices = []
        pca_labels = []
        for idx in range(min(rank, pca.components_.shape[0])):
            score_scale = np.nanstd(pca_scores_full[:, idx])
            if not np.isfinite(score_scale) or score_scale == 0:
                score_scale = 1.0
            pca_matrices.append(vector_to_landscape(score_scale * pca.components_[idx, :], X_scaled_df.columns))
            pca_labels.append(f"PC{idx + 1}")
        rank_corrs["PCA"] = program_correlation_from_landscape_matrices(pca_matrices, pca_labels)

        if cp_max_factors is not None:
            _, cp_time_rank, cp_space_rank = cp_max_factors
            cp_matrices = [np.outer(cp_time_rank[:, idx], cp_space_rank[:, idx]) for idx in range(rank)]
            cp_labels = [f"CP{idx + 1}" for idx in range(rank)]
            rank_corrs["CP"] = program_correlation_from_landscape_matrices(cp_matrices, cp_labels)
        else:
            rank_corrs["CP"] = None

        if vae_max_matrices:
            rank_corrs["VAE"] = program_correlation_from_landscape_matrices(
                vae_max_matrices[:rank],
                vae_max_labels[:rank]
            )
        else:
            rank_corrs["VAE"] = None

        cpvae_matrices = [np.outer(cpvae_max_time[:, idx], cpvae_max_space[:, idx]) for idx in range(rank)]
        cpvae_labels = [f"CPVAE{idx + 1}" for idx in range(rank)]
        rank_corrs["weighted CPVAE"] = program_correlation_from_landscape_matrices(cpvae_matrices, cpvae_labels)

        for model_name, corr_df in rank_corrs.items():
            if corr_df is None:
                continue
            rank_distinctness_corrs[(rank, model_name)] = corr_df
            corr_values = corr_df.values
            offdiag_mask = ~np.eye(corr_values.shape[0], dtype=bool)
            offdiag_values = corr_values[offdiag_mask]
            mean_abs_offdiag = float(np.nanmean(np.abs(offdiag_values))) if offdiag_values.size else np.nan
            max_abs_offdiag = float(np.nanmax(np.abs(offdiag_values))) if offdiag_values.size else np.nan
            rank_distinctness_summary_rows.append({
                "Rank": rank,
                "Model": model_name,
                "Mean_abs_offdiagonal_correlation": mean_abs_offdiag,
                "Max_abs_offdiagonal_correlation": max_abs_offdiag,
                "Distinctness_score_1_minus_mean_abs_corr": 1 - mean_abs_offdiag if np.isfinite(mean_abs_offdiag) else np.nan,
                "Basis_source": f"first_{rank}_programs_from_rank{rank_program_max_rank}_fit"
            })
            for row_label in corr_df.index:
                for col_label in corr_df.columns:
                    rank_distinctness_long_rows.append({
                        "Rank": rank,
                        "Model": model_name,
                        "Program_1": row_label,
                        "Program_2": col_label,
                        "Correlation": corr_df.loc[row_label, col_label],
                        "Basis_source": f"first_{rank}_programs_from_rank{rank_program_max_rank}_fit"
                    })

    rank_distinctness_long_df = pd.DataFrame(rank_distinctness_long_rows)
    rank_distinctness_summary_df = pd.DataFrame(rank_distinctness_summary_rows)
    rank_distinctness_long_df.to_csv(
        os.path.join(outdir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_program_distinctness_correlations_long.csv"),
        index=False
    )
    rank_distinctness_summary_df.to_csv(
        os.path.join(outdir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_program_distinctness_summary.csv"),
        index=False
    )

    if rank_distinctness_corrs:
        fig, axes = plt.subplots(
            len(rank_distinctness_values),
            len(rank_distinctness_models),
            figsize=(12.8, 2.05 * len(rank_distinctness_values)),
            squeeze=False
        )
        im = None
        for r_idx, rank in enumerate(rank_distinctness_values):
            for m_idx, model_name in enumerate(rank_distinctness_models):
                ax = axes[r_idx, m_idx]
                corr_df = rank_distinctness_corrs.get((rank, model_name))
                if corr_df is None:
                    ax.axis("off")
                    continue
                im = ax.imshow(corr_df.values, vmin=-1, vmax=1, cmap="coolwarm")
                if r_idx == 0:
                    ax.set_title(model_name, fontsize=10, pad=7)
                if m_idx == 0:
                    ax.set_ylabel(f"Rank {rank}", fontsize=9)
                ax.set_xticks(range(len(corr_df.columns)))
                ax.set_xticklabels(corr_df.columns, rotation=45, ha="right", fontsize=5.5)
                ax.set_yticks(range(len(corr_df.index)))
                ax.set_yticklabels(corr_df.index, fontsize=5.5)
                ax.tick_params(length=1.5, pad=1)
        fig.suptitle("Rank 2-10 program distinctness across PCA, CP, VAE, and weighted CPVAE", fontsize=13, y=0.982)
        fig.subplots_adjust(left=0.06, right=0.90, top=0.943, bottom=0.035, hspace=0.84, wspace=0.38)
        if im is not None:
            cbar_ax = fig.add_axes([0.925, 0.14, 0.014, 0.72])
            fig.colorbar(im, cax=cbar_ax, label="Program correlation")
        distinctness_pdf = os.path.join(outdir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_program_distinctness_correlation_heatmaps.pdf")
        plt.savefig(distinctness_pdf, dpi=300, bbox_inches="tight", transparent=True)
        plt.savefig(
            os.path.join(model_comparison_dir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_program_distinctness_correlation_heatmaps.pdf"),
            dpi=300,
            bbox_inches="tight",
            transparent=True
        )
        plt.close()
        rank_distinctness_long_df.to_csv(
            os.path.join(model_comparison_dir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_program_distinctness_correlations_long.csv"),
            index=False
        )
        rank_distinctness_summary_df.to_csv(
            os.path.join(model_comparison_dir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_program_distinctness_summary.csv"),
            index=False
        )

    benchmark_candidates = [
        os.path.join(model_comparison_dir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_standardized_model_metrics.csv"),
        os.path.join(outdir, "PCA_CP_VAE_CPVAE_rank2_to_rank10_standardized_model_metrics.csv")
    ]
    for benchmark_path in benchmark_candidates:
        if os.path.exists(benchmark_path):
            rank_model_benchmark_df = pd.read_csv(benchmark_path)
            break

    ########################################################
    # Best weighted-CPVAE-rank program landscapes
    ########################################################

    def select_best_cpvae_rank_from_benchmarks(metrics_df):
        cpvae_metrics = metrics_df[metrics_df["Model"].astype(str).str.upper().isin(["CPVAE", "WEIGHTED CPVAE", "WEIGHTED_CPVAE"])].copy()
        if cpvae_metrics.empty:
            return cpvae_rank, pd.DataFrame()
        metric_specs = [
            ("NRMSE_by_observed_sd", False),
            ("Pearson_r_observed_vs_reconstructed", True),
            ("Explained_variance", True),
            ("Strong_signal_ROC_AUC", True)
        ]
        score_parts = []
        for metric, higher_is_better in metric_specs:
            values = cpvae_metrics[metric].astype(float)
            span = values.max() - values.min()
            if not np.isfinite(span) or span == 0:
                scaled = pd.Series(1.0, index=cpvae_metrics.index)
            elif higher_is_better:
                scaled = (values - values.min()) / span
            else:
                scaled = (values.max() - values) / span
            cpvae_metrics[f"{metric}_scaled_for_selection"] = scaled
            score_parts.append(scaled)
        cpvae_metrics["Composite_CP_VAE_rank_score"] = pd.concat(score_parts, axis=1).mean(axis=1)
        best_row = cpvae_metrics.sort_values(["Composite_CP_VAE_rank_score", "Strong_signal_ROC_AUC"], ascending=[False, False]).iloc[0]
        return int(best_row["Rank"]), cpvae_metrics

    if rank_model_benchmark_df.empty:
        best_cpvae_landscape_rank = cpvae_rank
        cpvae_rank_selection_df = pd.DataFrame()
    else:
        best_cpvae_landscape_rank, cpvae_rank_selection_df = select_best_cpvae_rank_from_benchmarks(rank_model_benchmark_df)
    if not cpvae_rank_selection_df.empty:
        cpvae_rank_selection_df.to_csv(os.path.join(outdir, "weighted_CPVAE_best_rank_selection_scores.csv"), index=False)
        cpvae_rank_selection_df.to_csv(os.path.join(model_comparison_dir, "weighted_CPVAE_best_rank_selection_scores.csv"), index=False)

    ########################################################
    # Genes where CPVAE has the largest reconstruction advantage
    ########################################################

    cpvae_advantage_predictions = {
        "PCA": pca_max_prediction,
        "CP": cp_max_prediction,
        "VAE": vae_max_prediction,
        "CPVAE": cpvae_max_prediction
    }
    cpvae_advantage_predictions = {
        model_name: np.asarray(reconstructed, dtype=float)
        for model_name, reconstructed in cpvae_advantage_predictions.items()
        if reconstructed is not None
    }
    if {"PCA", "CP", "VAE", "CPVAE"}.issubset(cpvae_advantage_predictions.keys()):
        cpvae_gene_advantage_df = pd.DataFrame({"Gene": gene_ids})
        for model_name, reconstructed in cpvae_advantage_predictions.items():
            cpvae_gene_advantage_df[f"{model_name}_MSE"] = np.mean((X_scaled - reconstructed) ** 2, axis=1)
        comparator_cols = ["PCA_MSE", "CP_MSE", "VAE_MSE"]
        cpvae_gene_advantage_df["Best_non_CPVAE_MSE"] = cpvae_gene_advantage_df[comparator_cols].min(axis=1)
        cpvae_gene_advantage_df["Best_non_CPVAE_model"] = cpvae_gene_advantage_df[comparator_cols].idxmin(axis=1).str.replace("_MSE", "", regex=False)
        cpvae_gene_advantage_df["CPVAE_advantage_vs_best_other"] = cpvae_gene_advantage_df["Best_non_CPVAE_MSE"] - cpvae_gene_advantage_df["CPVAE_MSE"]
        for model_name in ["PCA", "CP", "VAE"]:
            cpvae_gene_advantage_df[f"CPVAE_advantage_vs_{model_name}"] = cpvae_gene_advantage_df[f"{model_name}_MSE"] - cpvae_gene_advantage_df["CPVAE_MSE"]
        cpvae_gene_advantage_df["CPVAE_rank_used"] = rank_program_max_rank
        cpvae_gene_advantage_df = cpvae_gene_advantage_df.merge(gene_summary, on="Gene", how="left")
        cpvae_gene_advantage_df["Gene_label"] = cpvae_gene_advantage_df["Gene_display"].fillna(cpvae_gene_advantage_df["Gene"])
        cpvae_gene_advantage_df.to_csv(os.path.join(outdir, "CPVAE_gene_reconstruction_error_advantage_over_PCA_CP_VAE.csv"), index=False)
        cpvae_gene_advantage_df.to_csv(os.path.join(model_comparison_dir, "CPVAE_gene_reconstruction_error_advantage_over_PCA_CP_VAE.csv"), index=False)

        top_advantage_genes = (
            cpvae_gene_advantage_df
            .sort_values("CPVAE_advantage_vs_best_other", ascending=False)
            .head(20)
            .copy()
        )
        top_advantage_genes["Plot_label"] = top_advantage_genes["Gene_label"].astype(str) + " (" + top_advantage_genes["Gene"].astype(str) + ")"
        plot_df = top_advantage_genes.iloc[::-1]
        fig = plt.figure(figsize=(11.5, 8.2))
        gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.45], wspace=0.16)
        ax_bar = fig.add_subplot(gs[0, 0])
        y_positions = np.arange(len(plot_df))
        ax_bar.barh(
            y_positions,
            plot_df["CPVAE_advantage_vs_best_other"],
            color="#2f7f74",
            alpha=0.88,
            label="vs best of PCA/CP/VAE"
        )
        for model_name, color in [("PCA", "#5b6fb9"), ("CP", "#c47a33"), ("VAE", "#8a5aa8")]:
            ax_bar.scatter(
                plot_df[f"CPVAE_advantage_vs_{model_name}"],
                y_positions,
                s=20,
                color=color,
                alpha=0.78,
                label=f"vs {model_name}"
            )
        ax_bar.axvline(0, color="black", linewidth=0.8)
        ax_bar.set_yticks(y_positions)
        ax_bar.set_yticklabels(plot_df["Plot_label"], fontsize=7)
        ax_bar.set_xlabel("MSE reduction by CPVAE")
        ax_bar.set_title("Largest CPVAE reconstruction-error advantage", fontsize=10, pad=10)
        ax_bar.legend(frameon=False, fontsize=7, loc="lower right")

        ax_heat = fig.add_subplot(gs[0, 1])
        mse_matrix = plot_df[["PCA_MSE", "CP_MSE", "VAE_MSE", "CPVAE_MSE"]].values.astype(float)
        row_means = mse_matrix.mean(axis=1, keepdims=True)
        row_stds = mse_matrix.std(axis=1, keepdims=True)
        row_stds[row_stds == 0] = 1.0
        mse_z = (mse_matrix - row_means) / row_stds
        im = ax_heat.imshow(mse_z, aspect="auto", cmap="RdYlBu_r", vmin=-1.8, vmax=1.8)
        ax_heat.set_xticks(range(4))
        ax_heat.set_xticklabels(["PCA", "CP", "VAE", "CPVAE"], fontsize=8)
        ax_heat.set_yticks(y_positions)
        ax_heat.set_yticklabels([])
        ax_heat.set_title("Per-gene relative reconstruction error", fontsize=10, pad=10)
        for yi, (_, row) in enumerate(plot_df.iterrows()):
            for xi, col in enumerate(["PCA_MSE", "CP_MSE", "VAE_MSE", "CPVAE_MSE"]):
                ax_heat.text(xi, yi, f"{row[col]:.2f}", ha="center", va="center", fontsize=6)
        cbar = fig.colorbar(im, ax=ax_heat, fraction=0.045, pad=0.02)
        cbar.set_label("Row-scaled MSE", fontsize=8)
        fig.suptitle(
            f"Genes where CPVAE has the largest reconstruction-error advantage (rank {rank_program_max_rank})",
            fontsize=12,
            y=0.98
        )
        fig.subplots_adjust(top=0.91, bottom=0.08, left=0.22, right=0.97)
        advantage_pdf = "CPVAE_top20_gene_reconstruction_error_advantage_over_PCA_CP_VAE.pdf"
        plt.savefig(os.path.join(outdir, advantage_pdf), dpi=300, bbox_inches="tight", transparent=True)
        plt.savefig(os.path.join(model_comparison_dir, advantage_pdf), dpi=300, bbox_inches="tight", transparent=True)
        plt.close()

        cpvae_advantage_example_genes = top_advantage_genes.head(6)["Gene"].to_list()
        reconstruction_panel_models = [
            ("Observed", X_scaled),
            ("PCA", cpvae_advantage_predictions["PCA"]),
            ("CP", cpvae_advantage_predictions["CP"]),
            ("VAE", cpvae_advantage_predictions["VAE"]),
            ("CPVAE", cpvae_advantage_predictions["CPVAE"])
        ]
        gene_to_index = {gene: idx for idx, gene in enumerate(gene_ids)}

        # Invert the gene-wise scaling and plot only raw-fitness landscapes.
        raw_reconstruction_panel_records = []
        for gene in cpvae_advantage_example_genes:
            gene_idx = gene_to_index[gene]
            gene_label = cpvae_gene_advantage_df.loc[cpvae_gene_advantage_df["Gene"] == gene, "Gene_label"].iloc[0]
            gene_mean = float(row_mean[gene_idx, 0]) if "row_mean" in globals() else float(np.nanmean(X_raw[gene_idx, :]))
            gene_std = float(row_std[gene_idx, 0]) if "row_std" in globals() else float(np.nanstd(X_raw[gene_idx, :]))
            if not np.isfinite(gene_std) or gene_std == 0:
                gene_std = 1.0
            for model_name, matrix_values in reconstruction_panel_models:
                model_vector_scaled = np.asarray(matrix_values, dtype=float)[gene_idx, :]
                model_vector_raw = model_vector_scaled * gene_std + gene_mean
                model_landscape_raw = vector_to_landscape(model_vector_raw, X_scaled_df.columns)
                for ti, time in enumerate(Full_Timepoints):
                    for si, space in enumerate(Spacepoints):
                        raw_reconstruction_panel_records.append({
                            "Gene": gene,
                            "Gene_label": gene_label,
                            "Model": model_name,
                            "Time": time,
                            "Space": space,
                            "Raw_fitness": model_landscape_raw[ti, si]
                        })
        raw_reconstruction_panel_df = pd.DataFrame(raw_reconstruction_panel_records)
        raw_reconstruction_panel_df.to_csv(os.path.join(outdir, "CPVAE_advantage_genes_observed_vs_reconstructed_raw_fitness_landscapes_long.csv"), index=False)
        raw_reconstruction_panel_df.to_csv(os.path.join(model_comparison_dir, "CPVAE_advantage_genes_observed_vs_reconstructed_raw_fitness_landscapes_long.csv"), index=False)

        fig, axes = plt.subplots(
            len(cpvae_advantage_example_genes),
            len(reconstruction_panel_models),
            figsize=(13.8, 2.05 * len(cpvae_advantage_example_genes)),
            squeeze=False
        )
        im = None
        for row_idx, gene in enumerate(cpvae_advantage_example_genes):
            gene_idx = gene_to_index[gene]
            gene_row = cpvae_gene_advantage_df.loc[cpvae_gene_advantage_df["Gene"] == gene].iloc[0]
            gene_label = gene_row["Gene_label"]
            raw_gene_values = raw_reconstruction_panel_df.loc[raw_reconstruction_panel_df["Gene"] == gene, "Raw_fitness"].values.astype(float)
            vmax = np.nanquantile(np.abs(raw_gene_values), 0.98)
            if not np.isfinite(vmax) or vmax == 0:
                vmax = 1.0
            for col_idx, (model_name, matrix_values) in enumerate(reconstruction_panel_models):
                ax = axes[row_idx, col_idx]
                sub = raw_reconstruction_panel_df[
                    (raw_reconstruction_panel_df["Gene"] == gene) &
                    (raw_reconstruction_panel_df["Model"] == model_name)
                ]
                model_landscape_raw = (
                    sub.pivot(index="Time", columns="Space", values="Raw_fitness")
                    .reindex(index=Full_Timepoints, columns=Spacepoints)
                    .values.astype(float)
                )
                im = ax.imshow(model_landscape_raw, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
                if row_idx == 0:
                    ax.set_title(model_name, fontsize=10, pad=7)
                if col_idx == 0:
                    advantage_value = gene_row["CPVAE_advantage_vs_best_other"]
                    ax.set_ylabel(f"{gene_label}\n{gene}\nadv={advantage_value:.2f}", fontsize=7)
                ax.set_xticks(range(len(Spacepoints)))
                ax.set_xticklabels(Spacepoints, rotation=45, ha="right", fontsize=6)
                ax.set_yticks(range(len(Full_Timepoints)))
                ax.set_yticklabels(Full_Timepoints, fontsize=6)
                if model_name != "Observed":
                    mse_col = f"{model_name}_MSE"
                    if mse_col in gene_row.index:
                        ax.text(
                            0.02,
                            0.96,
                            f"scaled MSE={gene_row[mse_col]:.2f}",
                            transform=ax.transAxes,
                            ha="left",
                            va="top",
                            fontsize=5.6,
                            bbox=dict(facecolor="white", alpha=0.62, edgecolor="none", pad=1.0)
                        )
        fig.suptitle(
            "Raw fitness landscapes for genes where CPVAE improves most over PCA, CP, and VAE",
            fontsize=12,
            y=0.985
        )
        fig.subplots_adjust(left=0.14, right=0.91, top=0.925, bottom=0.07, hspace=0.70, wspace=0.28)
        if im is not None:
            cbar_ax = fig.add_axes([0.93, 0.18, 0.014, 0.66])
            fig.colorbar(im, cax=cbar_ax, label="Beta(log2FC)")
        raw_reconstruction_panel_pdf = "CPVAE_advantage_genes_observed_vs_reconstructed_raw_fitness_landscapes.pdf"
        plt.savefig(os.path.join(outdir, raw_reconstruction_panel_pdf), dpi=300, bbox_inches="tight", transparent=True)
        plt.savefig(os.path.join(model_comparison_dir, raw_reconstruction_panel_pdf), dpi=300, bbox_inches="tight", transparent=True)
        plt.close()

    best_rank = best_cpvae_landscape_rank
    best_landscape_records = []
    best_landscape_rows = []
    if best_rank not in rank_cp_factors:
        cp_best_output = cp_reconstruction_for_four_model_rank(best_rank, return_factors=True)
        if cp_best_output is not None:
            _, _, rank_cp_factors[best_rank] = cp_best_output
    if best_rank not in rank_vae_models:
        vae_best_result = train_vanilla_vae_for_four_model_rank(best_rank, epochs=cpvae_scan_epochs, patience=3, return_model=True)
        if vae_best_result is not None:
            rank_vae_models[best_rank], _, rank_vae_latents[best_rank] = vae_best_result
    if best_rank not in rank_cpvae_models:
        rank_cpvae_models[best_rank], rank_cpvae_predictions[best_rank], _, _, _ = train_cp_structured_vae(best_rank, epochs=cpvae_scan_epochs, patience=3)

    cp_best_factors = rank_cp_factors.get(best_rank)
    vae_best_model = rank_vae_models.get(best_rank)
    vae_best_latent = rank_vae_latents.get(best_rank)
    cpvae_best_model = rank_cpvae_models[best_rank]
    cpvae_best_time = cpvae_best_model.time_factor.detach().cpu().numpy()
    cpvae_best_space = cpvae_best_model.space_factor.detach().cpu().numpy()

    vae_best_matrices, vae_best_labels = ([], [])
    if vae_best_model is not None:
        vae_best_matrices, vae_best_labels = decoded_vae_program_matrices(
            vae_best_model,
            getattr(vae_best_model, "latent_dim", best_rank),
            vae_best_latent,
            device if "device" in globals() else torch.device("cpu")
        )

    for idx in range(best_rank):
        row = []
        score_scale = np.nanstd(pca_scores_full[:, idx]) if idx < pca_scores_full.shape[1] else 1.0
        if not np.isfinite(score_scale) or score_scale == 0:
            score_scale = 1.0
        pca_matrix = vector_to_landscape(pca.mean_ + score_scale * pca.components_[idx, :], X_scaled_df.columns)
        row.append({"title": f"PCA PC{idx + 1}", "matrix": pca_matrix})
        for ti, time in enumerate(Full_Timepoints):
            for si, space in enumerate(Spacepoints):
                best_landscape_records.append({"Model": "PCA", "Rank": best_rank, "Program": f"PC{idx + 1}", "Time": time, "Space": space, "Value": pca_matrix[ti, si]})

        if cp_best_factors is not None:
            _, cp_best_time, cp_best_space = cp_best_factors
            cp_matrix = np.outer(cp_best_time[:, idx], cp_best_space[:, idx])
            row.append({"title": f"CP component {idx + 1}", "matrix": cp_matrix})
            for ti, time in enumerate(Full_Timepoints):
                for si, space in enumerate(Spacepoints):
                    best_landscape_records.append({"Model": "CP", "Rank": best_rank, "Program": f"CP{idx + 1}", "Time": time, "Space": space, "Value": cp_matrix[ti, si]})
        else:
            row.append(None)

        if idx < len(vae_best_matrices):
            vae_matrix = vae_best_matrices[idx]
            row.append({"title": f"VAE latent {idx + 1}", "matrix": vae_matrix})
            for ti, time in enumerate(Full_Timepoints):
                for si, space in enumerate(Spacepoints):
                    best_landscape_records.append({"Model": "VAE", "Rank": best_rank, "Program": f"VAE{idx + 1}", "Time": time, "Space": space, "Value": vae_matrix[ti, si]})
        else:
            row.append(None)

        cpvae_matrix = np.outer(cpvae_best_time[:, idx], cpvae_best_space[:, idx])
        row.append({"title": f"weighted CPVAE{idx + 1}", "matrix": cpvae_matrix})
        for ti, time in enumerate(Full_Timepoints):
            for si, space in enumerate(Spacepoints):
                best_landscape_records.append({"Model": "weighted_CPVAE", "Rank": best_rank, "Program": f"CPVAE{idx + 1}", "Time": time, "Space": space, "Value": cpvae_matrix[ti, si]})
        best_landscape_rows.append(row)

    best_landscape_base = "PCA_CP_VAE_CPVAE_best_CPVAE_rank_program_landscapes"
    best_rank_info_df = pd.DataFrame([{"Selected_CPVAE_rank": best_rank, "Selection_basis": "highest composite weighted-CPVAE benchmark score"}])
    best_rank_info_df.to_csv(os.path.join(outdir, "PCA_CP_VAE_CPVAE_best_CPVAE_selected_rank.csv"), index=False)
    best_rank_info_df.to_csv(os.path.join(model_comparison_dir, "PCA_CP_VAE_CPVAE_best_CPVAE_selected_rank.csv"), index=False)
    best_landscape_df = pd.DataFrame(best_landscape_records)
    best_landscape_df.to_csv(os.path.join(outdir, f"{best_landscape_base}_long.csv"), index=False)
    best_landscape_df.to_csv(os.path.join(model_comparison_dir, f"{best_landscape_base}_long.csv"), index=False)
    save_landscape_grid(
        best_landscape_rows,
        os.path.join(outdir, f"{best_landscape_base}.pdf"),
        f"Program landscapes at best weighted CPVAE benchmark rank ({best_rank})",
        value_label="Per-panel z-score of program loading",
        scale_mode="per_panel_zscore"
    )
    save_landscape_grid(
        best_landscape_rows,
        os.path.join(model_comparison_dir, f"{best_landscape_base}.pdf"),
        f"Program landscapes at best weighted CPVAE benchmark rank ({best_rank})",
        value_label="Per-panel z-score of program loading",
        scale_mode="per_panel_zscore"
    )

    # Also save rank-6, rank-7, and rank-8 views using the same fitted program basis.
    for requested_landscape_rank in [6, 7, 8]:
        if requested_landscape_rank <= len(best_landscape_rows):
            requested_rows = best_landscape_rows[:requested_landscape_rank]
            requested_base = f"PCA_CP_VAE_CPVAE_rank{requested_landscape_rank}_program_landscapes"
            requested_records = []
            for record in best_landscape_records:
                program_digits = "".join(ch for ch in str(record["Program"]) if ch.isdigit())
                if program_digits and int(program_digits) <= requested_landscape_rank:
                    requested_records.append(record)
            pd.DataFrame(requested_records).to_csv(os.path.join(outdir, f"{requested_base}_long.csv"), index=False)
            pd.DataFrame(requested_records).to_csv(os.path.join(model_comparison_dir, f"{requested_base}_long.csv"), index=False)
            save_landscape_grid(
                requested_rows,
                os.path.join(outdir, f"{requested_base}.pdf"),
                f"Rank-{requested_landscape_rank} program landscapes across PCA, CP, VAE, and weighted CPVAE",
                value_label="Per-panel z-score of program loading",
                scale_mode="per_panel_zscore"
            )
            save_landscape_grid(
                requested_rows,
                os.path.join(model_comparison_dir, f"{requested_base}.pdf"),
                f"Rank-{requested_landscape_rank} program landscapes across PCA, CP, VAE, and weighted CPVAE",
                value_label="Per-panel z-score of program loading",
                scale_mode="per_panel_zscore"
            )

    ########################################################
    # Four-model rank-6 performance comparison
    ########################################################

    four_model_predictions = {}
    four_model_predictions["PCA"] = pca_scores_full[:, :cpvae_rank] @ pca.components_[:cpvae_rank, :] + pca.mean_
    cp_rank_rec_for_four_model = cp_reconstruction_for_four_model_rank(cpvae_rank)
    if cp_rank_rec_for_four_model is not None:
        four_model_predictions["CP"] = cp_rank_rec_for_four_model
    vae_rank_rec_for_four_model = train_vanilla_vae_for_four_model_rank(cpvae_rank)
    if vae_rank_rec_for_four_model is not None:
        four_model_predictions["VAE"] = vae_rank_rec_for_four_model
    four_model_predictions["CPVAE"] = X_cpvae_rec

    four_model_metric_rows = []
    roc_threshold_quantile_four = 0.90
    roc_signal_threshold_four = np.nanquantile(np.abs(X_scaled).ravel(), roc_threshold_quantile_four)
    y_true_four = (np.abs(X_scaled).ravel() >= roc_signal_threshold_four).astype(int)

    for model_name, reconstructed in four_model_predictions.items():
        reconstructed = np.asarray(reconstructed, dtype=float)
        observed_flat = X_scaled.ravel()
        reconstructed_flat = reconstructed.ravel()
        residual_flat = observed_flat - reconstructed_flat
        y_score = np.abs(reconstructed_flat)
        fpr, tpr, thresholds = roc_curve(y_true_four, y_score)
        model_auc = auc(fpr, tpr)
        four_model_metric_rows.append({
            "Model": model_name,
            "Rank": cpvae_rank,
            "MSE": np.mean(residual_flat ** 2),
            "RMSE": np.sqrt(np.mean(residual_flat ** 2)),
            "MAE": np.mean(np.abs(residual_flat)),
            "NRMSE_by_observed_sd": np.sqrt(np.mean(residual_flat ** 2)) / np.nanstd(observed_flat),
            "Pearson_r_observed_vs_reconstructed": np.corrcoef(observed_flat, reconstructed_flat)[0, 1],
            "Explained_variance": 1 - np.var(residual_flat) / np.var(observed_flat),
            "Strong_signal_ROC_AUC": model_auc
        })
    four_model_metrics_df = pd.DataFrame(four_model_metric_rows)
    four_model_metrics_df.to_csv(os.path.join(outdir, "PCA_CP_VAE_CPVAE_rank6_standardized_model_metrics.csv"), index=False)

    metric_plot_df = four_model_metrics_df.set_index("Model")[[
        "NRMSE_by_observed_sd",
        "Pearson_r_observed_vs_reconstructed",
        "Explained_variance",
        "Strong_signal_ROC_AUC"
    ]]
    ax = metric_plot_df.plot(kind="bar", figsize=(8.8, 4.8))
    ax.set_ylabel("Metric value")
    ax.set_title("Four-model rank-6 standardized comparison")
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "PCA_CP_VAE_CPVAE_rank6_standardized_model_metrics.pdf"), dpi=300, bbox_inches="tight", transparent=True)
    plt.close()

    print("Saved CP-structured decoder VAE and four-model comparison outputs.")
else:
    print("CP-structured decoder VAE skipped because torch/tensor feature requirements are not available.")


Saved CP-structured decoder VAE and four-model comparison outputs.


In [ ]:
# %% Cell - CPVAE biology figures
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'

project = Path('/Users/sidaye/Documents/python/ST_MultiCAST')
outdir = project / 'Output' / 'model comparison'
ai_outdir = project / 'Output' / 'AI_spatiotemporal_models_python'
outdir.mkdir(parents=True, exist_ok=True)

spacepoints = ['st','SI1','SI2','SI3','SI4','SI5','SI6','SI7','SI8','SI9','ce','co']
timepoints = ['1h','3h','6h','12h','24h']
phase_map = {'1h':'Early','3h':'Early','6h':'Middle','12h':'Middle','24h':'Late'}
phase_colors = {'Early':'#e8f2ff','Middle':'#fff2cc','Late':'#ece7f2'}

landscape_path = outdir / 'PCA_CP_VAE_CPVAE_best_CPVAE_rank_program_landscapes_long.csv'
landscape = pd.read_csv(landscape_path)
cpvae_land = landscape[landscape['Model'].eq('weighted_CPVAE')].copy()
cpvae_land['Program_number'] = cpvae_land['Program'].str.extract(r'(\d+)').astype(int)
cpvae_land = cpvae_land.sort_values(['Program_number','Time','Space'])
programs = cpvae_land[['Program','Program_number']].drop_duplicates().sort_values('Program_number')
program_order = programs['Program'].tolist()

# ---------------- Temporal activation ----------------
temporal = (
    cpvae_land.assign(Abs_value=lambda d: d['Value'].abs())
    .groupby(['Program','Program_number','Time'], as_index=False)
    .agg(mean_abs_loading=('Abs_value','mean'), signed_mean_loading=('Value','mean'), max_abs_loading=('Abs_value','max'))
)
temporal['Phase'] = temporal['Time'].map(phase_map)
temporal.to_csv(outdir / 'CPVAE_best_rank_program_temporal_activation.csv', index=False)
temporal.to_csv(ai_outdir / 'CPVAE_best_rank_program_temporal_activation.csv', index=False)

heat = temporal.pivot(index='Program', columns='Time', values='mean_abs_loading').reindex(index=program_order, columns=timepoints)
row_scaled = heat.div(heat.max(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig = plt.figure(figsize=(12.8, 7.8))
gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.25], wspace=0.24)
ax_line = fig.add_subplot(gs[0,0])
x = np.arange(len(timepoints))
for i, prog in enumerate(program_order):
    vals = row_scaled.loc[prog].values.astype(float)
    ax_line.plot(x, vals, marker='o', linewidth=1.4, markersize=3.8, label=prog)
for phase in ['Early','Middle','Late']:
    idx = [i for i,t in enumerate(timepoints) if phase_map[t] == phase]
    ax_line.axvspan(min(idx)-0.45, max(idx)+0.45, color=phase_colors[phase], zorder=0)
    ax_line.text(np.mean(idx), 1.08, phase, ha='center', va='bottom', fontsize=8, color='#4b5563')
ax_line.set_xticks(x)
ax_line.set_xticklabels(timepoints)
ax_line.set_ylim(-0.04, 1.16)
ax_line.set_ylabel('Relative activation (row-normalized mean |loading|)')
ax_line.set_title('CPVAE program temporal activation lines', fontsize=10, pad=9)
ax_line.legend(frameon=False, fontsize=6.2, ncol=2, loc='upper left', bbox_to_anchor=(0.0, -0.12))

ax_heat = fig.add_subplot(gs[0,1])
im = ax_heat.imshow(row_scaled.values, aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax_heat.set_xticks(x)
ax_heat.set_xticklabels(timepoints)
ax_heat.set_yticks(range(len(program_order)))
ax_heat.set_yticklabels(program_order, fontsize=8)
ax_heat.set_title('Program x time activation heatmap', fontsize=10, pad=9)
for j,t in enumerate(timepoints):
    ax_heat.text(j, -0.88, phase_map[t], ha='center', va='center', fontsize=7, color='#4b5563')
for i in range(len(program_order)):
    peak_idx = int(np.nanargmax(row_scaled.iloc[i].values))
    ax_heat.text(peak_idx, i, '*', ha='center', va='center', color='white', fontsize=11, weight='bold')
cbar = fig.colorbar(im, ax=ax_heat, fraction=0.035, pad=0.02)
cbar.set_label('Relative activation', fontsize=8)
fig.suptitle('When each CPVAE hidden fitness program becomes important', fontsize=13, y=0.985)
fig.subplots_adjust(top=0.88, bottom=0.20, left=0.07, right=0.96)
for target in [outdir, ai_outdir]:
    fig.savefig(target / 'CPVAE_best_rank_program_temporal_activation.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.close(fig)

# ---------------- Spatial specificity ----------------
spatial = (
    cpvae_land.assign(Abs_value=lambda d: d['Value'].abs())
    .groupby(['Program','Program_number','Space'], as_index=False)
    .agg(mean_abs_loading=('Abs_value','mean'), signed_mean_loading=('Value','mean'), max_abs_loading=('Abs_value','max'))
)

def niche(space):
    if space == 'st': return 'stomach'
    if space.startswith('SI'): return 'small_intestine'
    if space == 'ce': return 'cecum'
    if space == 'co': return 'colon'
    return space
spatial['Niche'] = spatial['Space'].map(niche)
peak_space = spatial.sort_values('mean_abs_loading', ascending=False).groupby('Program').head(1)[['Program','Space','Niche','mean_abs_loading']]
peak_space = peak_space.rename(columns={'Space':'Peak_space','Niche':'Peak_niche','mean_abs_loading':'Peak_mean_abs_loading'})
spatial = spatial.merge(peak_space, on='Program', how='left')
spatial.to_csv(outdir / 'CPVAE_best_rank_program_spatial_specificity.csv', index=False)
spatial.to_csv(ai_outdir / 'CPVAE_best_rank_program_spatial_specificity.csv', index=False)

space_heat = spatial.pivot(index='Program', columns='Space', values='mean_abs_loading').reindex(index=program_order, columns=spacepoints)
space_scaled = space_heat.div(space_heat.max(axis=1).replace(0,np.nan), axis=0).fillna(0)
grouped = spatial.groupby(['Program','Program_number','Niche'], as_index=False).agg(mean_abs_loading=('mean_abs_loading','mean'))
group_order = ['stomach','small_intestine','cecum','colon']
group_mat = grouped.pivot(index='Program', columns='Niche', values='mean_abs_loading').reindex(index=program_order, columns=group_order).fillna(0)
group_scaled = group_mat.div(group_mat.sum(axis=1).replace(0,np.nan), axis=0).fillna(0)

fig = plt.figure(figsize=(13.6, 8.1))
gs = fig.add_gridspec(1, 2, width_ratios=[1.35, 0.95], wspace=0.28)
ax1 = fig.add_subplot(gs[0,0])
im = ax1.imshow(space_scaled.values, aspect='auto', cmap='magma', vmin=0, vmax=1)
ax1.set_xticks(range(len(spacepoints)))
ax1.set_xticklabels(spacepoints, rotation=45, ha='right')
ax1.set_yticks(range(len(program_order)))
ax1.set_yticklabels(program_order, fontsize=8)
ax1.set_title('CPVAE program spatial specificity', fontsize=10, pad=9)
for i, prog in enumerate(program_order):
    peak = peak_space[peak_space['Program'].eq(prog)].iloc[0]['Peak_space']
    j = spacepoints.index(peak)
    ax1.text(j, i, '*', ha='center', va='center', color='white', fontsize=11, weight='bold')
cbar = fig.colorbar(im, ax=ax1, fraction=0.035, pad=0.02)
cbar.set_label('Relative spatial specificity', fontsize=8)

ax2 = fig.add_subplot(gs[0,1])
left = np.zeros(len(program_order))
stack_colors = {'stomach':'#7aa6c2','small_intestine':'#76b77b','cecum':'#d99b56','colon':'#b36aa8'}
for group in group_order:
    vals = group_scaled[group].values
    ax2.barh(np.arange(len(program_order)), vals, left=left, color=stack_colors[group], label=group, height=0.74)
    left += vals
ax2.set_yticks(np.arange(len(program_order)))
ax2.set_yticklabels(program_order, fontsize=8)
ax2.invert_yaxis()
ax2.set_xlim(0, 1)
ax2.set_xlabel('Fraction of mean |loading|')
ax2.set_title('Grouped gut-region specificity', fontsize=10, pad=9)
ax2.legend(frameon=False, fontsize=7, loc='lower center', bbox_to_anchor=(0.5, -0.16), ncol=2)
for i, prog in enumerate(program_order):
    peak = peak_space[peak_space['Program'].eq(prog)].iloc[0]
    ax2.text(1.02, i, f"peak: {peak['Peak_space']}", va='center', fontsize=7, color='#374151')
fig.suptitle('Where each CPVAE hidden fitness program is spatially specific', fontsize=13, y=0.985)
fig.subplots_adjust(top=0.90, bottom=0.16, left=0.08, right=0.92)
for target in [outdir, ai_outdir]:
    fig.savefig(target / 'CPVAE_best_rank_program_spatial_specificity.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.close(fig)

# ---------------- Driver genes / TF / functional annotation ----------------
latent_path = ai_outdir / 'CPVAE_rank6_latent_gene_embedding.csv'
latent = pd.read_csv(latent_path)
program_cols = [c for c in latent.columns if re.fullmatch(r'CPVAE\d+', c)]
ann = pd.read_csv(project / 'Input' / 'new_annotations_with_uniprot_names.csv')
ann['locus_ID'] = ann['locus_ID'].astype(str)
tf = pd.read_excel(project / 'Input' / 'putative_transcription_regulators.xlsx')
tf_set = set(tf['locus_ID'].astype(str)) if 'locus_ID' in tf.columns else set()
ann_cols = [c for c in ann.columns if c != 'locus_ID']
ann['annotation_text'] = ann[ann_cols].astype(str).replace('nan','', regex=False).agg(' | '.join, axis=1)
ann_map = ann.set_index('locus_ID')['annotation_text'].to_dict()

category_patterns = {
    'TF/regulator': r'transcription|regulator|response regulator|sigma|repressor|activator|lysR|tetR|two-component',
    'Motility/chemotaxis': r'flagell|motility|chemotaxis|chemo|flg|fli|motA|motB|che[A-Z]',
    'Virulence/colonization': r'virulence|toxin|tcp|pilus|adhesin|hemolysin|colonization|biofilm|secretion|T6SS|type VI|type 6',
    'Stress/envelope': r'stress|heat shock|cold shock|oxidative|periplasm|envelope|cpx|rpo|chaperone|protease|detox',
    'Metabolism/transport': r'metabol|transport|permease|dehydrogenase|synthetase|synthase|oxidoreductase|kinase|ATP|ABC|PTS|amino|carbon|respiration',
}

def category_for(gene):
    text = ann_map.get(str(gene), '')
    low = text.lower()
    for cat, pat in category_patterns.items():
        if re.search(pat.lower(), low):
            return cat
    return 'Other/unknown'

records = []
for prog in program_cols:
    vals = latent[['Gene','Gene_display', prog]].copy()
    vals = vals.rename(columns={prog:'Loading'})
    vals['Abs_loading'] = vals['Loading'].abs()
    vals['Direction'] = np.where(vals['Loading'] >= 0, 'Positive', 'Negative')
    top_pos = vals.sort_values('Loading', ascending=False).head(12)
    top_neg = vals.sort_values('Loading', ascending=True).head(12)
    for _, row in pd.concat([top_pos, top_neg], axis=0).iterrows():
        gene = str(row['Gene'])
        display = row['Gene_display'] if pd.notna(row.get('Gene_display')) else gene
        records.append({
            'Program': prog,
            'Gene': gene,
            'Gene_display': display,
            'Loading': row['Loading'],
            'Abs_loading': abs(row['Loading']),
            'Direction': 'Positive' if row['Loading'] >= 0 else 'Negative',
            'Is_putative_TF': gene in tf_set,
            'Functional_category': 'TF/regulator' if gene in tf_set else category_for(gene),
            'Annotation_text': ann_map.get(gene, '')
        })
drivers = pd.DataFrame(records).drop_duplicates(['Program','Gene','Direction'])
drivers.to_csv(outdir / 'CPVAE_rank6_top_positive_negative_driver_genes_with_TF_labels.csv', index=False)
drivers.to_csv(ai_outdir / 'CPVAE_rank6_top_positive_negative_driver_genes_with_TF_labels.csv', index=False)

cat_colors = {
    'TF/regulator':'#8e44ad',
    'Motility/chemotaxis':'#1f77b4',
    'Virulence/colonization':'#d62728',
    'Stress/envelope':'#ff7f0e',
    'Metabolism/transport':'#2ca02c',
    'Other/unknown':'#7f7f7f'
}
fig, axes = plt.subplots(len(program_cols), 2, figsize=(14.8, 2.25*len(program_cols)), squeeze=False)
for r, prog in enumerate(program_cols):
    for c, direction in enumerate(['Positive','Negative']):
        ax = axes[r, c]
        sub = drivers[(drivers['Program'].eq(prog)) & (drivers['Direction'].eq(direction))].copy()
        sub = sub.sort_values('Loading', ascending=(direction=='Negative')).head(10)
        if direction == 'Positive':
            sub = sub.sort_values('Loading', ascending=True)
        else:
            sub = sub.sort_values('Loading', ascending=False)
        labels = []
        for _, row in sub.iterrows():
            label = str(row['Gene_display']) if pd.notna(row['Gene_display']) else row['Gene']
            if label == 'nan' or not label:
                label = row['Gene']
            if row['Is_putative_TF']:
                label = label + ' *TF'
            labels.append(label)
        y = np.arange(len(sub))
        colors = [cat_colors.get(cat, '#7f7f7f') for cat in sub['Functional_category']]
        ax.barh(y, sub['Loading'], color=colors, alpha=0.88)
        ax.axvline(0, color='black', lw=0.7)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f'{prog} top {direction.lower()} drivers', fontsize=9, pad=7)
        ax.set_xlabel('CPVAE latent gene loading')
        ax.tick_params(axis='x', labelsize=7)
        if c == 0:
            ax.set_ylabel(prog, fontsize=9, weight='bold')
legend_items = [Patch(facecolor=color, label=cat) for cat, color in cat_colors.items()]
fig.legend(handles=legend_items, frameon=False, fontsize=8, loc='lower center', ncol=3, bbox_to_anchor=(0.5, 0.01))
fig.suptitle('CPVAE identifies putative genetic drivers of each dynamic colonization program', fontsize=13, y=0.995)
fig.subplots_adjust(top=0.94, bottom=0.10, left=0.16, right=0.98, hspace=0.62, wspace=0.34)
for target in [outdir, ai_outdir]:
    fig.savefig(target / 'CPVAE_rank6_driver_genes_positive_negative_with_TF_labels.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.close(fig)

# compact driver dotplot
plot_top = drivers.copy()
plot_top['Signed_rank'] = plot_top.groupby(['Program','Direction'])['Abs_loading'].rank(method='first', ascending=False)
plot_top = plot_top[plot_top['Signed_rank'] <= 8].copy()
plot_top['Gene_label'] = plot_top['Gene_display'].fillna(plot_top['Gene']).astype(str)
plot_top.loc[plot_top['Is_putative_TF'], 'Gene_label'] += ' *TF'
plot_top['Program_num'] = plot_top['Program'].str.extract(r'(\d+)').astype(int)
plot_top = plot_top.sort_values(['Program_num','Direction','Abs_loading'], ascending=[True, True, False])
plot_top['Y_label'] = plot_top['Program'] + ' | ' + plot_top['Gene_label']
fig, ax = plt.subplots(figsize=(10.8, max(7.5, 0.18*len(plot_top))))
y = np.arange(len(plot_top))
colors = [cat_colors.get(cat, '#7f7f7f') for cat in plot_top['Functional_category']]
sizes = 20 + 90 * (plot_top['Abs_loading'] / plot_top['Abs_loading'].max())
ax.scatter(plot_top['Loading'], y, s=sizes, c=colors, alpha=0.86, edgecolor='black', linewidth=0.25)
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(y)
ax.set_yticklabels(plot_top['Y_label'], fontsize=6.2)
ax.invert_yaxis()
ax.set_xlabel('CPVAE latent gene loading')
ax.set_title('Top CPVAE driver genes and putative regulators', fontsize=12, pad=10)
fig.legend(handles=legend_items, frameon=False, fontsize=8, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.02))
fig.subplots_adjust(left=0.36, right=0.98, top=0.95, bottom=0.11)
for target in [outdir, ai_outdir]:
    fig.savefig(target / 'CPVAE_rank6_driver_gene_dotplot_with_TF_labels.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.close(fig)

print(outdir / 'CPVAE_best_rank_program_temporal_activation.pdf')
print(outdir / 'CPVAE_best_rank_program_spatial_specificity.pdf')
print(outdir / 'CPVAE_rank6_driver_genes_positive_negative_with_TF_labels.pdf')
print(outdir / 'CPVAE_rank6_driver_gene_dotplot_with_TF_labels.pdf')
